In [2]:
from optimization.acs import ACS_VRP,ACS_TSP
from optimization.bso import BSO_VRP,BSO_TSP
from optimization.ga import GA_VRP
from optimization.sa import SA_VRP
from optimization.tabu_search import TS_VRP
from optimization.dka import DKA_VRP
from optimization_time_dependent.dka import DKA_VRP as DKA_VRP_TD

from optimization.other.algorithm_woa import WOA_VRP

from optimization.aco import ACO_VRP,ACO_TSP
from optimization.haco import HACO_VRP,HACO_TSP

from optimization.koneksi import ConDB

import sql_connection
import json
import time
import random
import datetime
import copy
import pandas as pd
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.patches import FancyArrowPatch

import warnings
warnings.filterwarnings('ignore')

In [3]:
class GA_VRP_TD(object):
    def __init__(self,population_size = 50,crossover_rate=0.8,mutation_rate=0.2,max_iter = 300,max_idem = 15,random_state=None):
        self.db = ConDB()
        
        # parameter setting
        self.population_size = population_size #max size of population
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.max_iter = max_iter #max iteration of tabu search
        self.max_idem = max_idem #stop if the best fitness doesn't increase for max_idem iteration

        # set initial solution
        self.init_solution = [] #2D list of nodes, [[node1,node2,....],[node4,node5,....]]
        self.rest_nodes = [] #1D list of nodes, [node1,node2,node4,node5,....]

        # data model setting
        self.tour = None #POI yang dipilih oleh user untuk dikunjungi
        self.hotel = None #hotel yang dipilih oleh user
        self.timematrix = None
        self.max_travel_time = None
        self.travel_days = None
        
        #degree of interest (DOI for MAUT) setting
        self.degree_waktu = 1
        self.degree_tarif = 1
        self.degree_rating = 1
        self.degree_poi = 1
        self.degree_poi_penalty = 1
        self.degree_time_penalty = 1
        
        #scaler setting
        self.min_rating = None
        self.max_rating = None
        self.min_tarif = None
        self.max_tarif = None
        self.min_waktu = None
        self.max_waktu = None
        self.min_poi = None
        self.max_poi = None
        self.min_poi_penalty = None
        self.max_poi_penalty = None
        self.min_time_penalty = None
        self.max_time_penalty = None
        
        #set random seed
        if random_state != None:
            random.seed(random_state)
    
    def set_model(self,tour,hotel,timematrix,init_solution=[],start_day = 5,travel_days = 3, depart_time = datetime.time(8,0,0),max_travel_time = datetime.time(20,0,0),degree_waktu = 1,degree_tarif = 1,degree_rating = 1):
        #initiate model
        self.tour = copy.deepcopy(tour)
        self.hotel = copy.deepcopy(hotel)
        self.travel_days = travel_days
        self.hotel.depart_time = depart_time
        self.max_travel_time = max_travel_time
        self.timematrix = copy.deepcopy(timematrix)
        self.start_day = 6 # day index : monday - sunday => 0 - 6
        
        self.degree_waktu = degree_waktu
        self.degree_tarif = degree_tarif
        self.degree_rating = degree_rating
        
        self.min_rating = min([node.rating for node in self.tour])
        self.max_rating = max([node.rating for node in self.tour])
        self.min_tarif = min([node.tarif for node in self.tour])
        self.max_tarif = sum([node.tarif for node in self.tour])
        self.min_waktu = 0
        self.max_waktu = (self.diff_second_between_time(depart_time,max_travel_time))*self.travel_days
        self.min_poi = 0
        self.max_poi = len(self.tour)
        self.min_poi_penalty = 0
        self.max_poi_penalty = len(self.tour)
        self.min_time_penalty = 0
        self.max_time_penalty = ((24*3600)-self.diff_second_between_time(max_travel_time,depart_time))*travel_days
    
    def set_max_iter(self,max_iter):
        self.max_iter = max_iter
    
    def time_to_second(self,time):
        return (time.hour*3600)+(time.minute*60)+time.second
    
    def second_to_time(self,second):
        second = int(second)
        return datetime.time(second//3600,(second//60)%60,0) #ignore second detail
    
    def diff_second_between_time(self,time_a,time_b):
        #input: time_a and time_b, datetime.time()
        #output: time_b - time_a, seconds (int)
        return self.time_to_second(time_b) - self.time_to_second(time_a)
    
    def min_max_scaler(self,min_value,max_value,value):
        if max_value-min_value == 0:
            return 0
        else:
            return (value-min_value)/(max_value-min_value)
    
    def MAUT(self,solutions,consider_total_poi = True,use_penalty = True):
        #input: optimization solutions, format = [{"index":[],"waktu":[],"rating":[],"tarif":[]},...]
        #output: fitness value calculated using MAUT
        
        #concat all attribute lists (except for waktu)
        index_ls = sum([i['index'] for i in solutions],[])
        rating_ls = sum([i['rating'] for i in solutions],[])
        tarif_ls = sum([i['tarif'] for i in solutions],[])
        
        waktu_ls = [i['waktu'] for i in solutions]
        
        #rating
        avg_rating = sum(rating_ls)/len(rating_ls)
        score_rating = self.min_max_scaler(self.min_rating,self.max_rating,avg_rating)*self.degree_rating
        
        #tarif
        sum_tarif = sum(tarif_ls)
        score_tarif = (1-self.min_max_scaler(self.min_tarif,self.max_tarif,sum_tarif)) * self.degree_tarif
        
        #waktu
        waktu_per_day = [self.diff_second_between_time(i[0],i[-1]) for i in waktu_ls]
        sum_waktu = sum(waktu_per_day)
        score_waktu = (1-self.min_max_scaler(self.min_waktu,self.max_waktu,sum_waktu))*self.degree_waktu
        
        #poi
        count_poi = len(index_ls)
        score_poi = self.min_max_scaler(self.min_poi,self.max_poi,count_poi) if consider_total_poi == True else 0
        
        if use_penalty==True:
            #poi penalty
            penalty_index = [node._id for node in self.tour if node._id not in index_ls]
            count_penalty = len(penalty_index)
            score_poipenalty = (1-self.min_max_scaler(self.min_poi_penalty,self.max_poi_penalty,count_penalty)) * self.degree_poi_penalty
            
            #time penalty
            penalty_per_day = [max(self.diff_second_between_time(self.max_travel_time,i[-1]),0) for i in waktu_ls]
            sum_time_penalty = sum(penalty_per_day)
            score_timepenalty = (1-self.min_max_scaler(self.min_time_penalty,self.max_time_penalty,sum_time_penalty)) * self.degree_time_penalty
        else:
            score_poipenalty = 0
            score_timepenalty = 0
            
        #MAUT
        degree_rating = self.degree_rating
        degree_tarif = self.degree_tarif
        degree_waktu = self.degree_waktu
        degree_poi = self.degree_poi if consider_total_poi == True else 0
        degree_poi_penalty = self.degree_poi_penalty if use_penalty == True else 0
        degree_time_penalty = self.degree_time_penalty if use_penalty == True else 0

        pembilang = score_rating+score_tarif+score_waktu+score_poi+score_poipenalty+score_timepenalty
        penyebut = degree_rating+degree_tarif+degree_waktu+degree_poi+degree_poi_penalty+degree_time_penalty
        maut = pembilang/penyebut
        return maut
    
    def MAUT_between_two_nodes(self,current_node,next_node,jam,day_type):
        score_rating = self.degree_rating * self.min_max_scaler(self.min_rating,self.max_rating,next_node.rating)
        score_tarif = self.degree_tarif * (1-self.min_max_scaler(self.min_tarif,self.max_tarif,next_node.rating))
        score_waktu = self.degree_waktu * (1-self.min_max_scaler(self.min_waktu,self.max_waktu,self.timematrix[day_type][current_node._id][next_node._id][jam]))
        maut = (score_rating+score_tarif+score_waktu)/(self.degree_rating+self.degree_tarif+self.degree_waktu)
        return maut
    
    def next_node_check(self,current_node,next_node,jam,day_type):
        time_needed = self.time_to_second(current_node.depart_time)+self.timematrix[day_type][current_node._id][next_node._id][jam]+next_node.waktu_kunjungan
        time_limit = self.time_to_second(self.max_travel_time)
        if (time_needed <= time_limit) and (time_needed <= self.time_to_second(next_node.jam_tutup)):
            return True
        else:
            return False
    
    def set_next_node_depart_arrive_time(self,current_node,next_node,jam,day_type):
        arrive_time = self.time_to_second(current_node.depart_time)+self.timematrix[day_type][current_node._id][next_node._id][jam]
        arrive_time = max([arrive_time,self.time_to_second(next_node.jam_buka)])
        next_node.arrive_time = self.second_to_time(arrive_time)
        if next_node.tipe.lower() != "hotel":
            next_node.depart_time = self.second_to_time(arrive_time+next_node.waktu_kunjungan)
        return next_node
    
    def solution_list_of_nodes_to_dict(self, solutions):
        solution_dict = []
        day_index = self.start_day
        for day in solutions:
            day_type = "weekend" if day_index >= 5 else "weekday"
            day_solution = {"index":[],"waktu":[self.hotel.depart_time],"rating":[],"tarif":[]}
            day_solution['index'] = [i._id for i in day]
            day_solution['rating'] = [i.rating for i in day]
            day_solution['tarif'] = [i.tarif for i in day]
            
            last_waktu = self.time_to_second(day[-1].depart_time) + self.timematrix[day_type][day[-1]._id][self.hotel._id][day[-1].depart_time.hour]
            day_solution['waktu'].extend([i.arrive_time for i in day] + [self.second_to_time(last_waktu)])
            
            solution_dict.append(day_solution)
            day_index = day_index + 1 if day_index + 1 <= 6 else 0
        return solution_dict
    
    def split_itinerary(self,init_itinerary):
        final_solution = [] #2d list of nodes
        day = 1
        day_index = self.start_day
        tabu_nodes = [] #list of tabu node's id
        while day <= self.travel_days:
            current_node = self.hotel
            day_type = "weekend" if day_index >= 5 else "weekday"
            day_solution = [] #list of nodes
            next_node_candidates = [node for node in init_itinerary if node._id not in tabu_nodes]
            for i in range(len(next_node_candidates)):
                if self.next_node_check(current_node,next_node_candidates[i],current_node.depart_time.hour,day_type):
                    next_node_candidates[i] = self.set_next_node_depart_arrive_time(current_node,next_node_candidates[i],current_node.depart_time.hour,day_type)
                    day_solution.append(next_node_candidates[i])
                    tabu_nodes.append(next_node_candidates[i]._id)
                    current_node = next_node_candidates[i]
            
            if len(day_solution) > 0:
                final_solution.append(day_solution)
            
            if len(tabu_nodes) == len(self.tour):
                break
            
            day += 1
            day_index = day_index + 1 if day_index + 1 <= 6 else 0
        return final_solution
    
    def partially_mapped_crossover(self,parent1,parent2): #PMX
        start,end = sorted(random.sample(range(len(parent1)), 2))
        child1 = [None] * len(parent1)
        child2 = copy.deepcopy(child1)
        mapping1 = {parent1[i]._id:parent2[i] for i in range(start,end+1)}
        mapping2 = {parent2[i]._id:parent1[i] for i in range(start,end+1)}

        for i in range(len(parent1)):
            if start <= i <= end:
                child1[i],child2[i] = parent1[i],parent2[i] 
            else:
                gene1,gene2 = parent2[i],parent1[i]
                while gene1._id in mapping1:
                    gene1 = mapping1[gene1._id]
                while gene2._id in mapping2:
                    gene2 = mapping2[gene2._id]
                child1[i],child2[i] = gene1,gene2

        return child1,child2
    
    def swap_mutation(self,individual):
        individu = copy.deepcopy(individual)
        pos1,pos2 = random.sample(range(len(individu)),2)
        individu[pos1],individu[pos2] = individu[pos2],individu[pos1]
        return individu
    
    def construct_solution(self):
        solution_dict = {}
        fitness = 0
        
        idem_counter = 0
        population = [random.sample(self.tour,len(self.tour)) for i in range(self.population_size)]
        for i in range(self.max_iter):
                        
            #crossover and mutation
            offspring = []
            for ind in range(0,self.population_size,2):
                parent1,parent2 = population[ind],population[ind+1]

                #crossover
                if random.uniform(0,1) < self.crossover_rate:
                    child1,child2 = self.partially_mapped_crossover(parent1,parent2)
                else:
                    child1,child2 = copy.deepcopy(parent1),copy.deepcopy(parent2)
                
                #mutation
                if random.uniform(0,1) < self.mutation_rate:
                    child1 = self.swap_mutation(child1)
                if random.uniform(0,1) < self.mutation_rate:
                    child2 = self.swap_mutation(child2)
                
                offspring.append(child1)
                offspring.append(child2)
            
            #selection roulette wheel
            all_individuals = population + offspring
            fitness_values = [self.MAUT(self.solution_list_of_nodes_to_dict(self.split_itinerary(individual))) for individual in all_individuals]
            
            population = random.choices(all_individuals,weights=np.array(fitness_values)/sum(fitness_values),k=self.population_size)

            best_solution = max(population,key=lambda x: self.MAUT(self.solution_list_of_nodes_to_dict(self.split_itinerary(x))))
            best_solution_dict = self.solution_list_of_nodes_to_dict(self.split_itinerary(best_solution))
            best_fitness = self.MAUT(best_solution_dict)
            
            if fitness < best_fitness:
                fitness = best_fitness
                solution_dict = best_solution_dict
                idem_counter = 0
            else:
                idem_counter += 1
                if idem_counter > self.max_idem:
                    return solution_dict,fitness
        
        return solution_dict,fitness

# Time Dependent Time Matrix Initial Preprocessing

In [19]:
# timematrix = pd.read_csv("raw_data/place_timematrix_time_dependent.csv")

In [21]:
# timematrix

In [22]:
# timematrix = timematrix.drop(["id","durasi"],axis = 1)

# for col in timematrix.columns[2:]:
#     timematrix[col] = timematrix[col]*3600

In [23]:
# timematrix = timematrix.set_index(["id_a","id_b"])

#how to find specific travel time between 2 POIs
# timematrix.loc[1].loc[10][00]

In [24]:
# timematrix

In [25]:
# timematrix.to_csv("raw_data/place_timematrix_time_dependent_prep.csv")

In [4]:
#get all POIs and hotels id and name
query = """SELECT 
            p.post_id,
            p.post_title_id
        FROM 
            posts p
        """

df_all_tours = sql_connection.read_from_sql(query)

In [72]:
timematrix_weekday = pd.read_csv("raw_data/weekday_timematrix_fix.csv")
timematrix_weekend = pd.read_csv("raw_data/weekend_timematrix_fix.csv")

In [73]:
#handle id_a (source)
timematrix_weekday = timematrix_weekday.merge(df_all_tours.rename(columns={"post_id":"id_a_2","post_title_id":"name_a"}),
                                              on = "name_a",
                                              how = "inner")

timematrix_weekday["id_a"] = timematrix_weekday["id_a_2"]

timematrix_weekend = timematrix_weekend.merge(df_all_tours.rename(columns={"post_id":"id_a_2","post_title_id":"name_a"}),
                                              on = "name_a",
                                              how = "inner")

timematrix_weekend["id_a"] = timematrix_weekend["id_a_2"]

#handle id_b (destination)
timematrix_weekday = timematrix_weekday.merge(df_all_tours.rename(columns={"post_id":"id_b_2","post_title_id":"name_b"}),
                                              on = "name_b",
                                              how = "inner")

timematrix_weekday["id_b"] = timematrix_weekday["id_b_2"]

timematrix_weekend = timematrix_weekend.merge(df_all_tours.rename(columns={"post_id":"id_b_2","post_title_id":"name_b"}),
                                              on = "name_b",
                                              how = "inner")

timematrix_weekend["id_b"] = timematrix_weekend["id_b_2"]

#delete unused columns
timematrix_weekday = timematrix_weekday.drop(["id_a_2","id_b_2"],axis=1)
timematrix_weekend = timematrix_weekend.drop(["id_a_2","id_b_2"],axis=1)

In [74]:
timematrix_weekday.tail()

,id,id_a,id_b,day_type,slot_time,durasi_base,durasi_minutes,relation_type,area_relation,distance_category,...,type_a,name_b,type_b,currentTravelTime,freeFlowTravelTime,factor,factorToBase,lengthInMeters,trafficDelayInSeconds,status
157075,10473,185,1,weekday,17:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1302.0,1302.0,1.0,1.276471,6195.0,0.0,OK
157076,10473,185,1,weekday,18:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1262.0,1262.0,1.0,1.237255,6195.0,0.0,OK
157077,10473,185,1,weekday,19:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1173.0,1173.0,1.0,1.150000,6195.0,0.0,OK
157078,10473,185,1,weekday,20:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1085.0,1085.0,1.0,1.063725,6195.0,0.0,OK
157079,10473,185,1,weekday,21:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1004.0,1004.0,1.0,0.984314,6195.0,0.0,OK


In [75]:
timematrix_weekend.tail()

,id,id_a,id_b,day_type,slot_time,durasi_base,durasi_minutes,relation_type,area_relation,distance_category,...,type_a,name_b,type_b,currentTravelTime,freeFlowTravelTime,factor,factorToBase,lengthInMeters,trafficDelayInSeconds,status
157075,10473,185,1,weekend,17:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1286.0,1286.0,1.000000,1.260784,6404.0,0.0,OK
157076,10473,185,1,weekend,18:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1276.0,1276.0,1.000000,1.250980,6404.0,0.0,OK
157077,10473,185,1,weekend,19:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1421.0,1421.0,1.000000,1.393137,6122.0,0.0,OK
157078,10473,185,1,weekend,20:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1215.0,1215.0,1.000000,1.191176,6404.0,0.0,OK
157079,10473,185,1,weekend,21:00,1020,17.0,hotel_to_location,outer_to_center,medium,...,hotel,MALIOBORO JOGJAKARTA,location,1553.0,1031.0,1.506305,1.522549,6195.0,522.0,OK


In [76]:
timematrix_weekday["id_a"].unique()

array([  1,   3,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,
        17,  18,  19,  20,  21,  22,  23,  24,  26,  32,  33,  34,  35,
        36,  37,  38,  39,  41,  44,  45,  46,  47,  48,  50,  51,  52,
        53,  55,  56,  57,  59,  60,  62,  63,  64,  65,  66,  68,  69,
        70,  71,  72,  73,  74,  75,  77,  78,  79,  81,  82,  83,  84,
        85,  86,  88,  89,  91,  92,  93,  94,  96,  97,  98, 100, 103,
       106, 109, 112, 114, 115, 117, 122, 125, 127, 128, 136, 139, 141,
       143, 146, 147, 149, 154, 155, 157, 162, 164, 166, 172, 174, 175,
       176, 185,   2], dtype=int64)

In [80]:
timematrix_weekday.to_csv("raw_data/weekday_timematrix_prep.csv",index=False)
timematrix_weekend.to_csv("raw_data/weekend_timematrix_prep.csv",index=False)

# Generate Data

In [5]:
def generate_data(n = 30, random_state = None):
    random.seed(random_state)
    query = """SELECT 
                    p.post_id,
                    p.post_type,
                    CASE
                        WHEN pj.pj_jam_buka = pj.pj_jam_tutup THEN "tutup"
                        ELSE "buka"
                    END AS is_operate
                FROM 
                    posts p
                LEFT JOIN
                    posts_jadwal pj
                    ON p.post_id = pj.pj_id_tempat AND pj.pj_hari = "minggu"
                WHERE
                    p.post_id IN ( 1,   3,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,
                                    17,  18,  19,  20,  21,  22,  23,  24,  26,  32,  33,  34,  35,
                                    36,  37,  38,  39,  41,  44,  45,  46,  47,  48,  50,  51,  52,
                                    53,  55,  56,  57,  59,  60,  62,  63,  64,  65,  66,  68,  69,
                                    70,  71,  72,  73,  74,  75,  77,  78,  79,  81,  82,  83,  84,
                                    85,  86,  88,  89,  91,  92,  93,  94,  96,  97,  98, 100, 103,
                                   106, 109, 112, 114, 115, 117, 122, 125, 127, 128, 136, 139, 141,
                                   143, 146, 147, 149, 154, 155, 157, 162, 164, 166, 172, 174, 175,
                                   176, 185,   2)
                """

    df_location = sql_connection.read_from_sql(query)
    
    tourid = df_location[(df_location['post_type']=="location")&
                         (df_location['is_operate'] != "tutup")]['post_id'].values.tolist()
    tourid = random.sample(tourid,n)
    idhotel = df_location[df_location['post_type']=="hotel"]['post_id'].values.tolist()
    idhotel = idhotel[random.randint(0,len(idhotel)-1)]
    
    db = ConDB()

    hotel = db.HotelbyID(idhotel)
    tur = db.WisatabyID(tourid)
    timematrix = db.TimeMatrixbyID(hotel._id,tourid)
    
    return hotel,tur,timematrix

In [6]:
# timematrix = pd.read_csv("raw_data/place_timematrix_time_dependent_prep.csv")
# timematrix = timematrix.set_index(["id_a","id_b"])

In [7]:
def load_time_matrix(filepath,hotel,tur):
    """
    Reads a time matrix CSV file and returns a nested dict:
    { id_a: { id_b: { slot_hour: currentTravelTime } } }
    """
    df = pd.read_csv(filepath, usecols=["id_a", "id_b", "slot_time", "currentTravelTime"])
    df = df[(df["id_a"].isin([node._id for node in tur]+[hotel._id]))&
            (df["id_b"].isin([node._id for node in tur]+[hotel._id]))]
 
    df["id_a"] = df["id_a"].astype(int)
    df["id_b"] = df["id_b"].astype(int)
    df["slot_hour"] = df["slot_time"].str.split(":").str[0].astype(int)
 
    matrix = {}
    for id_a, group_a in df.groupby("id_a"):
        matrix[id_a] = {
            id_b: dict(zip(group_b["slot_hour"], group_b["currentTravelTime"]))
            for id_b, group_b in group_a.groupby("id_b")
        }
 
    return matrix

def build_travel_time_dict(weekday_path, weekend_path, hotel, tur):
    return {
        "weekday": load_time_matrix(weekday_path, hotel, tur),
        "weekend": load_time_matrix(weekend_path, hotel, tur),
    }

# Analisis


In [8]:
def find_node(node_id,tour):
    for node in tour:
        if node._id == node_id:
            return node
    return False

def time_to_second(time):
    return (time.hour*3600)+(time.minute*60)+time.second

def diff_second_between_time(time_a,time_b):
    #input: time_a and time_b, datetime.time()
    #output: time_b - time_a, seconds (int)
    return time_to_second(time_b) - time_to_second(time_a)

def get_attribute_analysis(solution):
    days_needed = len(solution)
    poi_included = len(sum([sol["index"] for sol in solution],[]))
    avg_rating = sum(sum([sol["rating"] for sol in solution],[]))/poi_included
    total_tarif = sum(sum([sol["tarif"] for sol in solution],[]))
    total_duration = sum([diff_second_between_time(j[0],j[-1]) for j in [i['waktu'] for i in solution]])
    return days_needed,poi_included,avg_rating,total_tarif,total_duration

def get_analysis(model,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day = 5,return_solution=False):
    start = time.time()
    model.set_model(tour=tur,hotel=hotel,timematrix=timematrix,
                    start_day=start_day,
                    travel_days=travel_days,
                    degree_waktu = dwaktu,degree_tarif = dtarif,degree_rating = drating)
    solution,fitness = model.construct_solution()[-2:]
    end = time.time()
    run_time = end-start
    
    daily_fitness = []
    for day in solution:
        daily_fitness.append(model.MAUT([day],use_penalty=False))
    days_needed,poi_included,avg_rating,total_tarif,total_duration = get_attribute_analysis(solution)
    results = {"solution":[sol["index"] for sol in solution],
               "fitness":fitness,
               "daily_fitness":daily_fitness,
               "days_needed":days_needed,
               "poi_included":poi_included,
               "avg_rating":avg_rating,
               "total_tarif":total_tarif,
               "total_duration":total_duration,
               "run_time":run_time}
    
    if return_solution == True:
        return results,solution
    else:
        return results

In [9]:
def get_detail_node_on_solution(solution):
    db = ConDB()
    node_solution = []
    for i in solution:
        nodes = db.WisatabyID(i)
        node_solution.append(nodes)
    return node_solution

def visualize_solution(solution_node,hotel):
    # Sample VRP data
    vrp_data = [[hotel]+sol+[hotel] for sol in solution_node]
    
    # Generate random node coordinates
    node_coords = {node._id: (float(node.long), float(node.lat)) for node in set(node for route in vrp_data for node in route)}  

    # Number of routes (you can calculate this)
    num_routes = len(vrp_data)

    # Create the plot
    fig, ax = plt.subplots(figsize=(12,9))

    # Define the colors for routes using a colormap
    cmap = plt.get_cmap('tab20', num_routes)

    for i, route in enumerate(vrp_data):
        x = [node_coords[node._id][0] for node in route]
        y = [node_coords[node._id][1] for node in route]
        color = cmap(i)  # Get the color for the route
        # Plot the nodes (scatter plot)
        ax.scatter(x, y, c=color, label=f'Route {i + 1}', s=100)

        for j in range(len(x) - 1):
            # Make the arrow heads larger by increasing mutation_scale
            arrow = FancyArrowPatch(
                (x[j], y[j]), (x[j + 1], y[j + 1]),
                arrowstyle='->', mutation_scale=15, color=color
            )
            ax.add_patch(arrow)

    # Set node 159 (start and end point) to black
    start_x, start_y = node_coords[hotel._id]
    ax.scatter(start_x, start_y, c='k', label='Hotel', s=100)

    # Add labels for nodes
    for node, (x, y) in node_coords.items():
        ax.annotate(str(node), (x, y), textcoords="offset points", xytext=(0, 10), ha='center')

    # Add legend
    ax.legend()

    plt.show()



In [10]:
# def time_dependent_timematrix(timematrix,hotel,tur):
#     timematrix_new = timematrix[(timematrix["id_a"].isin([node._id for node in tur]+[hotel._id]))&
#                                 (timematrix["id_b"].isin([node._id for node in tur]+[hotel._id]))]
#     timematrix_new = timematrix_new.set_index(["id_a","id_b"])
    
#     timematrix_dict = {}
    
#     for node_a in tur+[hotel]:
#         timematrix_dict[node_a._id] = {}
#         for node_b in tur+[hotel]:
#             if node_a._id == node_b._id:
#                 continue
#             timematrix_dict[node_a._id][node_b._id] = {}
#             for hour in timematrix_new.columns:
#                 timematrix_dict[node_a._id][node_b._id][int(hour)] = timematrix_new.loc[node_a._id].loc[node_b._id][hour]
    
#     return timematrix_dict


In [11]:
# timematrix_new

In [55]:
[i._id for i in tur]

[4,
 5,
 7,
 8,
 13,
 15,
 17,
 20,
 24,
 28,
 29,
 34,
 35,
 38,
 39,
 43,
 44,
 47,
 57,
 60,
 62,
 72,
 81,
 84,
 85,
 89,
 90,
 92,
 96,
 99]

In [63]:
timematrix = pd.read_csv("raw_data/weekday_timematrix_fix.csv")
timematrix[(timematrix["name_a"]=="Affandi Museum")]["id_a"]

6360      61
6361      61
6362      61
6363      61
6364      61
          ..
153069    61
153070    61
153071    61
153072    61
153073    61
Name: id_a, Length: 1590, dtype: int64

In [59]:
timematrix.head()

,id,id_a,id_b,day_type,slot_time,durasi_base,durasi_minutes,relation_type,area_relation,distance_category,...,type_a,name_b,type_b,currentTravelTime,freeFlowTravelTime,factor,factorToBase,lengthInMeters,trafficDelayInSeconds,status
0,2,1,2,weekday,07:00,134,2.233333,location_to_location,center_to_center,short,...,location,Tourism Zone Malioboro,location,124.0,124.0,1.0,0.925373,630.0,0.0,OK
1,3,1,3,weekday,07:00,605,10.083333,location_to_location,center_to_center,medium,...,location,Plengkung Gading,location,613.0,613.0,1.0,1.013223,2956.0,0.0,OK
2,4,1,4,weekday,07:00,228,3.800000,location_to_location,center_to_center,short,...,location,Fort Vredeburg Museum,location,199.0,199.0,1.0,0.872807,918.0,0.0,OK
3,5,1,5,weekday,07:00,222,3.700000,location_to_location,center_to_center,short,...,location,Titik Nol Km Yogyakarta,location,210.0,210.0,1.0,0.945946,983.0,0.0,OK
4,6,1,6,weekday,07:00,256,4.266667,location_to_location,center_to_center,short,...,location,Museum Sonobudoyo Unit I,location,233.0,233.0,1.0,0.910156,1088.0,0.0,OK


In [12]:
weekday_file = "raw_data/weekday_timematrix_prep.csv"
weekend_file = "raw_data/weekend_timematrix_prep.csv"

In [13]:
hotel,tur,timematrix_old = generate_data(n=30,random_state=40)

# timematrix_new = timematrix[(timematrix["id_a"].isin([node._id for node in tur]+[hotel._id]))&
#                            (timematrix["id_b"].isin([node._id for node in tur]+[hotel._id]))]
# timematrix_new = timematrix_new.set_index(["id_a","id_b"])

# timematrix_new = time_dependent_timematrix(timematrix,hotel,tur)
timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
travel_days = 3
dwaktu,dtarif,drating = (1,1,1)

127


In [14]:
# dka_vrp = DKA_VRP(random_state=100)
# results_dka_vrp,solution_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix_old,travel_days,dwaktu,dtarif,drating,return_solution=True)
# solution_dka_vrp

[{'index': [4, 8, 89, 85, 44, 92, 96, 7],
  'waktu': [datetime.time(8, 0),
   datetime.time(8, 13),
   datetime.time(9, 49),
   datetime.time(11, 53),
   datetime.time(13, 33),
   datetime.time(14, 40),
   datetime.time(16, 3),
   datetime.time(16, 47),
   datetime.time(18, 54),
   datetime.time(20, 2)],
  'rating': [4.6, 4.7, 4.8, 4.4, 4.7, 4.5, 4.7, 4.8],
  'tarif': [0, 3000, 0, 15000, 0, 0, 0, 0]},
 {'index': [29, 39, 35, 15, 13, 24, 43],
  'waktu': [datetime.time(8, 0),
   datetime.time(8, 54),
   datetime.time(12, 10),
   datetime.time(14, 11),
   datetime.time(15, 14),
   datetime.time(15, 55),
   datetime.time(17, 16),
   datetime.time(18, 39),
   datetime.time(20, 32)],
  'rating': [4.3, 4.6, 4.6, 4.5, 4.8, 4.4, 4.6],
  'tarif': [30000, 6000, 10000, 0, 0, 0, 0]},
 {'index': [20, 99, 81, 60, 17, 34, 57],
  'waktu': [datetime.time(8, 0),
   datetime.time(9, 0),
   datetime.time(10, 38),
   datetime.time(12, 20),
   datetime.time(13, 7),
   datetime.time(14, 12),
   datetime.time(

In [94]:
# dka_vrp_td = DKA_VRP_TD(random_state=100)
# results_dka_vrp_td,solution_dka_vrp_td = get_analysis(dka_vrp_td,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,return_solution=True)
# solution_dka_vrp_td

In [95]:
# [i._id for i in tur]
# hotel._id

In [14]:
ga_vrp_td = GA_VRP_TD(random_state=100)
results_ga_vrp_td,solution_ga_vrp_td = get_analysis(ga_vrp_td,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,return_solution=True)
solution_ga_vrp_td

[{'index': [81, 46, 88, 57, 15, 32, 21, 52, 66, 83],
  'waktu': [datetime.time(8, 0),
   datetime.time(8, 15),
   datetime.time(9, 3),
   datetime.time(10, 49),
   datetime.time(12, 17),
   datetime.time(13, 47),
   datetime.time(14, 27),
   datetime.time(16, 18),
   datetime.time(16, 51),
   datetime.time(17, 45),
   datetime.time(18, 49),
   datetime.time(19, 59)],
  'rating': [4.6, 5.0, 4.8, 4.5, 4.5, 4.5, 4.6, 4.8, 4.5, 4.3],
  'tarif': [0, 0, 0, 0, 0, 0, 0, 0, 0, 20000]},
 {'index': [38, 14, 79, 50, 7, 10, 24],
  'waktu': [datetime.time(8, 0),
   datetime.time(9, 0),
   datetime.time(11, 41),
   datetime.time(12, 53),
   datetime.time(15, 41),
   datetime.time(16, 44),
   datetime.time(17, 45),
   datetime.time(19, 13),
   datetime.time(19, 56)],
  'rating': [4.4, 4.5, 4.6, 4.8, 4.8, 4.6, 4.4],
  'tarif': [8000, 0, 3000, 0, 0, 0, 0]},
 {'index': [11, 44, 68, 6, 20, 96, 2],
  'waktu': [datetime.time(8, 0),
   datetime.time(8, 17),
   datetime.time(9, 19),
   datetime.time(10, 35),


In [15]:
results_ga_vrp_td

{'solution': [[81, 46, 88, 57, 15, 32, 21, 52, 66, 83],
  [38, 14, 79, 50, 7, 10, 24],
  [11, 44, 68, 6, 20, 96, 2]],
 'fitness': 0.6340343915343915,
 'daily_fitness': [0.5858300264550264, 0.5637537792894938, 0.5750647203325776],
 'days_needed': 3,
 'poi_included': 24,
 'avg_rating': 4.6125,
 'total_tarif': 49000,
 'total_duration': 129240,
 'run_time': 0.77152419090271}

## Hyperparameter tuning

### Iteration convergence

In [5]:
iteration_convergence_analysis ={
    "1,1,1": [],
    "1,0,0": [],
    "0,1,0": [],
    "0,0,1": [],
}

doi_combi = [(1,1,1),(1,0,0),(0,1,0),(0,0,1)]

for doi in doi_combi:
    hotel,tur,timematrix = generate_data(n=30,random_state=40)
    travel_days = 3
    dwaktu,dtarif,drating = doi
    
    # test hyperparameter max_iter : 
    dka_vrp = DKA_VRP(random_state=100,return_fitness_history=True)
    dka_vrp.set_model(tour=tur,hotel=hotel,timematrix=timematrix,
                    travel_days=travel_days,
                    degree_waktu = dwaktu,degree_tarif = dtarif,degree_rating = drating)
    solution,fitness,fitness_history = dka_vrp.construct_solution()
    
    iteration_convergence_analysis[",".join([str(i) for i in doi])] = fitness_history

In [6]:
iteration_convergence_analysis

{'1,1,1': [0.5967983728736419,
  0.5967983728736419,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.5985987843962753,
  0.6000769945572814,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6077828981054787,
  0.6104948938954318,
  0.612501066734938,
  0.612501066734938,
  0.612501066734938,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6135813136485179,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6197314392430879,
  0.6199629207245694,
  0.

In [7]:
result_df = pd.DataFrame()
for i in iteration_convergence_analysis:
    temp_df = pd.DataFrame({'best_fitness':iteration_convergence_analysis[i]})
    temp_df['doi_combi'] = i
    temp_df = temp_df.reset_index()
    temp_df = temp_df.rename(columns={'index':'iter'})
    temp_df['iter'] = temp_df['iter']+1
    result_df = pd.concat([result_df,temp_df]) 

In [8]:
result_df

,iter,best_fitness,doi_combi
0,1,0.596798,"1,1,1"
1,2,0.596798,"1,1,1"
2,3,0.598599,"1,1,1"
3,4,0.598599,"1,1,1"
4,5,0.598599,"1,1,1"
...,...,...,...
104,105,0.786639,"0,0,1"
105,106,0.786639,"0,0,1"
106,107,0.786639,"0,0,1"
107,108,0.786639,"0,0,1"


In [9]:
result_df.to_excel("dka_iteration_convergence.xlsx",index=False)

### Tuning for hyperparameters

In [12]:
hotel,tur,timematrix = generate_data(n=30,random_state=40)
travel_days = 3
dwaktu,dtarif,drating = 1,1,1

#### Hyperparameter : n

In [6]:
# n
n_ls = [5,10,15,20]
n_results = []
for n in n_ls:
    print(n)
    dka_vrp = DKA_VRP(n=n,random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    n_results.append([results_dka_vrp['fitness'],results_dka_vrp['run_time']])

5
10
15
20


In [21]:
n_results

[[0.6434002510167384, 2.2668440341949463],
 [0.6666653680328295, 6.990387439727783],
 [0.6646505376344088, 17.57559585571289],
 [0.6738713297853086, 60.89896059036255]]

In [24]:
result = {"n":n_ls,
          "fitness":[i[0] for i in n_results],
          "run_time":[i[1] for i in n_results]}
result = pd.DataFrame(result)
result.to_excel("dka_tuning_n_parameter.xlsx",index=False)

#### Hyperparameter : p

In [19]:
# p
p_ls = [0.25,0.5,0.75]
p_results = []
for p in p_ls:
    print(p)
    dka_vrp = DKA_VRP(p=p,random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    p_results.append([results_dka_vrp['fitness'],results_dka_vrp['run_time']])

0.25
0.5
0.75


In [20]:
p_results

[[0.6434002510167384, 2.1020259857177734],
 [0.6434002510167384, 2.304880142211914],
 [0.6390237500452555, 1.3925681114196777]]

#### Hyperparameter : smep

In [23]:
# smep
smep_ls = [3,5,8]
smep_results = []
for smep in smep_ls:
    print(smep)
    dka_vrp = DKA_VRP(p = 0.25,smep=smep,random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    smep_results.append([results_dka_vrp['fitness'],results_dka_vrp['run_time']])

3
5
8


In [24]:
smep_results

[[0.6557834166033091, 2.144763708114624],
 [0.6434002510167384, 2.234604597091675],
 [0.6642810271170486, 1.7997057437896729]]

## Outlier Sensitivity

In [10]:
outlier = pd.read_csv('Hasil Clustering Eps 1500.csv')
outlier.head()

,Unnamed: 0,id,name,type,latitude,longitude,tarif,durasi,rating,dbscan_cluster_relaxed1500,is_outlier
0,0,1,MALIOBORO JOGJAKARTA,location,-7.792576,110.365843,0,5400,4.8,0,False
1,1,2,Tourism Zone Malioboro,location,-7.798249,110.365279,0,3600,4.7,0,False
2,2,3,Plengkung Gading,location,-7.813728,110.362907,0,1800,4.6,0,False
3,3,4,Bringin Kembar Yogyakarta,location,-7.811936,110.363194,0,5400,4.6,0,False
4,4,5,Rumah Hantu Malioboro,location,-7.795638,110.365354,35000,2700,4.5,0,False


In [11]:
id_outlier = outlier[outlier['is_outlier']==True]['id'].values

In [12]:
id_outlier

array([39, 58, 62, 64, 65], dtype=int64)

In [13]:
hotel,tur,timematrix = generate_data(n=87,random_state=40)
dwaktu,dtarif,drating = 1,0,0

In [32]:
num_outlier = 0
travel_days = 0

outlier_sensitivity_analysis = {}
while num_outlier < len(id_outlier):
    travel_days += 1
    dka_vrp = DKA_VRP(random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    solution_ls = sum(results_dka_vrp['solution'],[])
    num_outlier = len([outlier for outlier in id_outlier if outlier in solution_ls])
    
    outlier_sensitivity_analysis[travel_days] = {"dka_vrp":num_outlier}

In [37]:
outlier_sensitivity_analysis

In [8]:
outlier_sensitivity_analysis = {1: {'dka_vrp': 0},
 2: {'dka_vrp': 0},
 3: {'dka_vrp': 0},
 4: {'dka_vrp': 0},
 5: {'dka_vrp': 0},
 6: {'dka_vrp': 1},
 7: {'dka_vrp': 3},
 8: {'dka_vrp': 2},
 9: {'dka_vrp': 3},
 10: {'dka_vrp': 2},
 11: {'dka_vrp': 4},
 12: {'dka_vrp': 4},
 13: {'dka_vrp': 5}}

In [16]:
min_days = 1
max_days = 13

In [14]:
for n_days in outlier_sensitivity_analysis:
    travel_days = n_days
    
    acs_vrp = ACS_VRP(random_state=100)
    results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_acs_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["acs_vrp"] = num_outlier
    
    aco_vrp = ACO_VRP(random_state=100)
    results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_aco_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["aco_vrp"] = num_outlier
    
    bso_vrp = BSO_VRP(random_state=100)
    results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_bso_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["bso_vrp"] = num_outlier
    
    ga_vrp = GA_VRP(random_state=100)
    results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_ga_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["ga_vrp"] = num_outlier
    
    ts_vrp = TS_VRP(random_state=100)
    results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_ts_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["ts_vrp"] = num_outlier
    
    sa_vrp = SA_VRP(random_state=100)
    results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
    num_outlier = len([outlier for outlier in id_outlier if outlier in sum(results_sa_vrp['solution'],[])])
    outlier_sensitivity_analysis[travel_days]["sa_vrp"] = num_outlier

In [15]:
outlier_sensitivity_analysis

{1: {'dka_vrp': 0,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 0,
  'ga_vrp': 0,
  'ts_vrp': 0,
  'sa_vrp': 0},
 2: {'dka_vrp': 0,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 0,
  'ga_vrp': 0,
  'ts_vrp': 0,
  'sa_vrp': 0},
 3: {'dka_vrp': 0,
  'acs_vrp': 0,
  'aco_vrp': 2,
  'bso_vrp': 0,
  'ga_vrp': 0,
  'ts_vrp': 0,
  'sa_vrp': 0},
 4: {'dka_vrp': 0,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 0,
  'ga_vrp': 0,
  'ts_vrp': 0,
  'sa_vrp': 0},
 5: {'dka_vrp': 0,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 0,
  'ga_vrp': 1,
  'ts_vrp': 0,
  'sa_vrp': 0},
 6: {'dka_vrp': 1,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 0,
  'ga_vrp': 2,
  'ts_vrp': 0,
  'sa_vrp': 1},
 7: {'dka_vrp': 3,
  'acs_vrp': 2,
  'aco_vrp': 0,
  'bso_vrp': 3,
  'ga_vrp': 3,
  'ts_vrp': 0,
  'sa_vrp': 1},
 8: {'dka_vrp': 2,
  'acs_vrp': 0,
  'aco_vrp': 0,
  'bso_vrp': 3,
  'ga_vrp': 1,
  'ts_vrp': 0,
  'sa_vrp': 3},
 9: {'dka_vrp': 3,
  'acs_vrp': 3,
  'aco_vrp': 2,
  'bso_vrp': 3,
  'ga_vrp': 2,
  'ts_vrp': 0,

In [17]:
result = pd.DataFrame()
for i in outlier_sensitivity_analysis:
    if i not in range(min_days,max_days+1):
        continue
    else:
        travel_days = i
        methods = []
        num_outlier = []
        for j in outlier_sensitivity_analysis[i]:
            methods.append(j)
            num_outlier.append(outlier_sensitivity_analysis[i][j])
        temp_res = pd.DataFrame({'methods':methods,
                                'num_outlier':num_outlier})
        temp_res['travel_days'] = travel_days
        result = pd.concat([result,temp_res])

In [18]:
result

,methods,num_outlier,travel_days
0,dka_vrp,0,1
1,acs_vrp,0,1
2,aco_vrp,0,1
3,bso_vrp,0,1
4,ga_vrp,0,1
...,...,...,...
2,aco_vrp,5,13
3,bso_vrp,4,13
4,ga_vrp,5,13
5,ts_vrp,5,13


In [19]:
result.to_excel('dka_outlier_sensitivity.xlsx',index=False)

## General Performance (50 random nodes)

In [96]:
def result_to_xlsx_rd_nodes(result,random_ls,filename):
    df = pd.DataFrame()
    for method in result:
        res_method = result[method]
        df_method = {
            "random_seed" : random_ls,
            "fitness" : [],
            "fitness_1" : [],
            "fitness_2" : [],
            "fitness_3" : [],
            "poi_included" : [],
            "rating" : [],
            "tarif" : [],
            "total_duration" : [],
            "run_time" : [],
        }
        for res in res_method:
            df_method["fitness"].append(res['fitness'])
            df_method["fitness_1"].append(res['daily_fitness'][0])
            df_method["fitness_2"].append(res['daily_fitness'][1])
            df_method["fitness_3"].append(res['daily_fitness'][2])
            df_method["poi_included"].append(res['poi_included'])
            df_method["rating"].append(res['avg_rating'])
            df_method["tarif"].append(res['total_tarif'])
            df_method["total_duration"].append(res['total_duration'])
            df_method["run_time"].append(res['run_time'])
        df_method = pd.DataFrame(df_method)
        df_method['method'] = method
        df = pd.concat([df,df_method])
    df.to_excel(filename,index=False)

### All DOI 1

In [102]:
random_ls = list(range(5,(5*50)+1,5))
random_node_analysis ={
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
for i in range(len(random_ls)):
    # setting
    hotel,tur,_ = generate_data(n=30,random_state=random_ls[i]) #random state 50 blacklist
    timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
    start_day = 5 #start from saturday
    travel_days = 3
    dwaktu,dtarif,drating = 1,1,1
    
    print("random state = ",random_ls[i])
    
#     acs_vrp = ACS_VRP(random_state=100)
#     results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["acs_vrp"].append(results_acs_vrp)
    
#     aco_vrp = ACO_VRP(random_state=100)
#     results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["aco_vrp"].append(results_aco_vrp)
    
#     bso_vrp = BSO_VRP(random_state=100)
#     results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["bso_vrp"].append(results_bso_vrp)
    
#     ga_vrp = GA_VRP(random_state=100)
#     results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ga_vrp"].append(results_ga_vrp)

#     ts_vrp = TS_VRP(random_state=100)
#     results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ts_vrp"].append(results_ts_vrp)

#     sa_vrp = SA_VRP(random_state=100)
#     results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["sa_vrp"].append(results_sa_vrp)
    
    dka_vrp = DKA_VRP_TD_2(random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
    random_node_analysis["dka_vrp"].append(results_dka_vrp)

141
random state =  5
141
random state =  10
125
random state =  15
115
random state =  20
109
random state =  25
149
random state =  30
127
random state =  35
127
random state =  40
162
random state =  45
174
random state =  50
162
random state =  55
154
random state =  60
172
random state =  65
143
random state =  70
115
random state =  75
114
random state =  80
139
random state =  85
157
random state =  90
136
random state =  95
174
random state =  100
103
random state =  105
109
random state =  110
172
random state =  115
139
random state =  120
162
random state =  125
127
random state =  130
122
random state =  135
115
random state =  140
185
random state =  145
100
random state =  150
103
random state =  155
172
random state =  160
155
random state =  165
114
random state =  170
115
random state =  175
136
random state =  180
128
random state =  185
136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  205
127
random state =  210
139
random s

In [103]:
random_node_analysis

{'dka_vrp': [{'solution': [[6, 81, 50, 21, 11, 89, 16, 53, 9, 36],
    [65, 79, 14, 18, 13, 10],
    [64, 55, 88, 44, 1, 7]],
   'fitness': 0.6206339963823783,
   'daily_fitness': [0.5817444264653003,
    0.5444590524991011,
    0.5898485257101762],
   'days_needed': 3,
   'poi_included': 22,
   'avg_rating': 4.659090909090909,
   'total_tarif': 71000,
   'total_duration': 122460,
   'run_time': 0.7825324535369873},
  {'solution': [[7, 74, 32, 52, 36, 88, 86, 24, 57, 3],
    [65, 33, 22, 44, 2, 13],
    [20, 34, 11, 94, 92, 21, 46]],
   'fitness': 0.6447699114250147,
   'daily_fitness': [0.611524470899471, 0.5732835894405042, 0.566633999935672],
   'days_needed': 3,
   'poi_included': 23,
   'avg_rating': 4.613043478260869,
   'total_tarif': 30000,
   'total_duration': 127560,
   'run_time': 1.165229082107544},
  {'solution': [[7, 79, 39, 24, 26, 2, 36],
    [6, 94, 96, 10, 35, 17, 21, 32],
    [62, 60, 68, 18, 16, 92, 74]],
   'fitness': 0.617630604211884,
   'daily_fitness': [0.60829

In [104]:
result_to_xlsx_rd_nodes(random_node_analysis,random_ls,"dka_random_nodes_alldoi_1_TD.xlsx")

### Travel Duration 1, other 0

In [105]:
random_ls = list(range(5,(5*50)+1,5))
random_node_analysis ={
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
for i in range(len(random_ls)):
    # setting
    hotel,tur,_ = generate_data(n=30,random_state=random_ls[i]) #random state 50 blacklist
    timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
    start_day = 5 #start from saturday
    travel_days = 3
    dwaktu,dtarif,drating = 1,0,0
    
    print("random state = ",random_ls[i])
    
#     acs_vrp = ACS_VRP(random_state=100)
#     results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["acs_vrp"].append(results_acs_vrp)
    
#     aco_vrp = ACO_VRP(random_state=100)
#     results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["aco_vrp"].append(results_aco_vrp)
    
#     bso_vrp = BSO_VRP(random_state=100)
#     results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["bso_vrp"].append(results_bso_vrp)
    
#     ga_vrp = GA_VRP(random_state=100)
#     results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ga_vrp"].append(results_ga_vrp)

#     ts_vrp = TS_VRP(random_state=100)
#     results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ts_vrp"].append(results_ts_vrp)

#     sa_vrp = SA_VRP(random_state=100)
#     results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["sa_vrp"].append(results_sa_vrp)
    
    dka_vrp = DKA_VRP_TD_2(random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
    random_node_analysis["dka_vrp"].append(results_dka_vrp)

141
random state =  5
141
random state =  10
125
random state =  15
115
random state =  20
109
random state =  25
149
random state =  30
127
random state =  35
127
random state =  40
162
random state =  45
174
random state =  50
162
random state =  55
154
random state =  60
172
random state =  65
143
random state =  70
115
random state =  75
114
random state =  80
139
random state =  85
157
random state =  90
136
random state =  95
174
random state =  100
103
random state =  105
109
random state =  110
172
random state =  115
139
random state =  120
162
random state =  125
127
random state =  130
122
random state =  135
115
random state =  140
185
random state =  145
100
random state =  150
103
random state =  155
172
random state =  160
155
random state =  165
114
random state =  170
115
random state =  175
136
random state =  180
128
random state =  185
136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  205
127
random state =  210
139
random s

In [106]:
random_node_analysis

{'dka_vrp': [{'solution': [[7, 9, 65, 16, 44, 21, 33, 88],
    [81, 18, 14, 1, 6, 11, 36, 86, 50, 10],
    [45, 84, 55, 89, 13, 53]],
   'fitness': 0.6576388888888889,
   'daily_fitness': [0.467824074074074,
    0.5064814814814814,
    0.4409722222222222],
   'days_needed': 3,
   'poi_included': 24,
   'avg_rating': 4.645833333333332,
   'total_tarif': 81000,
   'total_duration': 125640,
   'run_time': 2.3658127784729004},
  {'solution': [[36, 88, 20, 34, 57, 22, 2],
    [44, 86, 18, 11, 97, 13, 52, 24, 7, 21],
    [72, 94, 66, 33, 32, 92, 3, 46]],
   'fitness': 0.6673225308641976,
   'daily_fitness': [0.4511574074074074,
    0.5023148148148148,
    0.46504629629629624],
   'days_needed': 3,
   'poi_included': 25,
   'avg_rating': 4.579999999999999,
   'total_tarif': 150000,
   'total_duration': 129120,
   'run_time': 0.42519116401672363},
  {'solution': [[7, 6, 68, 60, 24, 35, 96, 21, 36],
    [16, 74, 92, 20, 18, 2, 17, 10],
    [39, 71, 75, 33, 83]],
   'fitness': 0.625,
   'daily_f

In [107]:
result_to_xlsx_rd_nodes(random_node_analysis,random_ls,"dka_random_nodes_dwaktu_1_TD.xlsx")

### Tarif 1, other 0

In [108]:
random_ls = list(range(5,(5*50)+1,5))
random_node_analysis ={
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
for i in range(len(random_ls)):
    # setting
    hotel,tur,_ = generate_data(n=30,random_state=random_ls[i]) #random state 50 blacklist
    timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
    start_day = 5 #start from saturday
    travel_days = 3
    dwaktu,dtarif,drating = 0,1,0
    
    print("random state = ",random_ls[i])
    
#     acs_vrp = ACS_VRP(random_state=100)
#     results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["acs_vrp"].append(results_acs_vrp)
    
#     aco_vrp = ACO_VRP(random_state=100)
#     results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["aco_vrp"].append(results_aco_vrp)
    
#     bso_vrp = BSO_VRP(random_state=100)
#     results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["bso_vrp"].append(results_bso_vrp)
    
#     ga_vrp = GA_VRP(random_state=100)
#     results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ga_vrp"].append(results_ga_vrp)

#     ts_vrp = TS_VRP(random_state=100)
#     results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ts_vrp"].append(results_ts_vrp)

#     sa_vrp = SA_VRP(random_state=100)
#     results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["sa_vrp"].append(results_sa_vrp)
    
    dka_vrp = DKA_VRP_TD_2(random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
    random_node_analysis["dka_vrp"].append(results_dka_vrp)

141
random state =  5
141
random state =  10
125
random state =  15
115
random state =  20
109
random state =  25
149
random state =  30
127
random state =  35
127
random state =  40
162
random state =  45
174
random state =  50
162
random state =  55
154
random state =  60
172
random state =  65
143
random state =  70
115
random state =  75
114
random state =  80
139
random state =  85
157
random state =  90
136
random state =  95
174
random state =  100
103
random state =  105
109
random state =  110
172
random state =  115
139
random state =  120
162
random state =  125
127
random state =  130
122
random state =  135
115
random state =  140
185
random state =  145
100
random state =  150
103
random state =  155
172
random state =  160
155
random state =  165
114
random state =  170
115
random state =  175
136
random state =  180
128
random state =  185
136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  205
127
random state =  210
139
random s

In [109]:
random_node_analysis

{'dka_vrp': [{'solution': [[6, 64, 50, 21, 33, 89, 9],
    [11, 16, 79, 53, 18, 13, 1],
    [14, 55, 34, 36, 81, 86, 88, 10]],
   'fitness': 0.8092233009708738,
   'daily_fitness': [0.5551779935275081,
    0.5713592233009709,
    0.625242718446602],
   'days_needed': 3,
   'poi_included': 22,
   'avg_rating': 4.631818181818182,
   'total_tarif': 71000,
   'total_duration': 126480,
   'run_time': 0.7704958915710449},
  {'solution': [[44, 33, 84, 86, 88, 46, 74],
    [18, 34, 94, 36, 52, 57, 66, 7],
    [11, 92, 13, 3, 2, 32, 20, 21, 24, 22]],
   'fitness': 0.8736890924612556,
   'daily_fitness': [0.5528368794326242,
    0.6333333333333333,
    0.6453900709219859],
   'days_needed': 3,
   'poi_included': 25,
   'avg_rating': 4.604,
   'total_tarif': 40000,
   'total_duration': 128880,
   'run_time': 1.1668100357055664},
  {'solution': [[17, 96, 18, 35, 79, 71, 21],
    [94, 6, 39, 10, 32, 33, 36],
    [68, 20, 16, 60, 92, 7, 74, 24]],
   'fitness': 0.808926448335495,
   'daily_fitness': 

In [110]:
result_to_xlsx_rd_nodes(random_node_analysis,random_ls,"dka_random_nodes_dtarif_1_TD.xlsx")

### Rating 1, other 0

In [111]:
random_ls = list(range(5,(5*50)+1,5))
random_node_analysis ={
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
for i in range(len(random_ls)):
    # setting
    hotel,tur,_ = generate_data(n=30,random_state=random_ls[i]) #random state 50 blacklist
    timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
    start_day = 5 #start from saturday
    travel_days = 3
    dwaktu,dtarif,drating = 0,0,1
    
    print("random state = ",random_ls[i])
    
#     acs_vrp = ACS_VRP(random_state=100)
#     results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["acs_vrp"].append(results_acs_vrp)
    
#     aco_vrp = ACO_VRP(random_state=100)
#     results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["aco_vrp"].append(results_aco_vrp)
    
#     bso_vrp = BSO_VRP(random_state=100)
#     results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["bso_vrp"].append(results_bso_vrp)
    
#     ga_vrp = GA_VRP(random_state=100)
#     results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ga_vrp"].append(results_ga_vrp)

#     ts_vrp = TS_VRP(random_state=100)
#     results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["ts_vrp"].append(results_ts_vrp)

#     sa_vrp = SA_VRP(random_state=100)
#     results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#     random_node_analysis["sa_vrp"].append(results_sa_vrp)
    
    dka_vrp = DKA_VRP_TD_2(random_state=100)
    results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
    random_node_analysis["dka_vrp"].append(results_dka_vrp)

141
random state =  5
141
random state =  10
125
random state =  15
115
random state =  20
109
random state =  25
149
random state =  30
127
random state =  35
127
random state =  40
162
random state =  45
174
random state =  50
162
random state =  55
154
random state =  60
172
random state =  65
143
random state =  70
115
random state =  75
114
random state =  80
139
random state =  85
157
random state =  90
136
random state =  95
174
random state =  100
103
random state =  105
109
random state =  110
172
random state =  115
139
random state =  120
162
random state =  125
127
random state =  130
122
random state =  135
115
random state =  140
185
random state =  145
100
random state =  150
103
random state =  155
172
random state =  160
155
random state =  165
114
random state =  170
115
random state =  175
136
random state =  180
128
random state =  185
136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  205
127
random state =  210
139
random s

In [112]:
random_node_analysis

{'dka_vrp': [{'solution': [[21, 50, 1, 84, 55, 75, 33, 36],
    [79, 64, 7, 10, 53, 9],
    [44, 45, 14, 89, 11, 81, 16, 88, 13]],
   'fitness': 0.7456521739130431,
   'daily_fitness': [0.341666666666666,
    0.37777777777777766,
    0.35370370370370235],
   'days_needed': 3,
   'poi_included': 23,
   'avg_rating': 4.669565217391304,
   'total_tarif': 146000,
   'total_duration': 127680,
   'run_time': 1.2605645656585693},
  {'solution': [[34, 66, 18, 11, 97, 2, 32, 36],
    [86, 57, 44, 20, 88, 71, 13],
    [72, 52, 94, 24, 7, 33, 21, 92, 3, 46]],
   'fitness': 0.7666666666666664,
   'daily_fitness': [0.2761904761904763,
    0.3513605442176868,
    0.3880952380952384],
   'days_needed': 3,
   'poi_included': 25,
   'avg_rating': 4.579999999999999,
   'total_tarif': 155000,
   'total_duration': 128220,
   'run_time': 0.8085999488830566},
  {'solution': [[62, 92, 7, 68, 60, 24, 96, 36],
    [6, 20, 71, 10, 2, 74],
    [12, 75, 18, 17, 21, 35, 33]],
   'fitness': 0.7595238095238094,
   '

In [113]:
result_to_xlsx_rd_nodes(random_node_analysis,random_ls,"dka_random_nodes_drating_1_TD.xlsx")

## Random DOI Combinations

In [114]:
def result_to_xlsx_rd_doi(result,random_doi,random_ls,filename):
    df_doi = pd.DataFrame()
    for doi_res in range(len(result)):
        df = pd.DataFrame()
        for method in result[doi_res]:
            res_method = result[doi_res][method]
            df_method = {
                "random_seed" : random_ls,
                "fitness" : [],
                "fitness_1" : [],
                "fitness_2" : [],
                "fitness_3" : [],
                "poi_included" : [],
                "rating" : [],
                "tarif" : [],
                "total_duration" : [],
                "run_time" : [],
            }
            for res in res_method:
                df_method["fitness"].append(res['fitness'])
                df_method["fitness_1"].append(res['daily_fitness'][0])
                df_method["fitness_2"].append(res['daily_fitness'][1])
                df_method["fitness_3"].append(res['daily_fitness'][2])
                df_method["poi_included"].append(res['poi_included'])
                df_method["rating"].append(res['avg_rating'])
                df_method["tarif"].append(res['total_tarif'])
                df_method["total_duration"].append(res['total_duration'])
                df_method["run_time"].append(res['run_time'])
            df_method = pd.DataFrame(df_method)
            df_method['method'] = method
            df = pd.concat([df,df_method])
        
        doi = random_doi[doi_res]
        df['dwaktu'] = doi[0]
        df['dtarif'] = doi[1]
        df['drating'] = doi[2]
        df = df[['dwaktu','dtarif','drating','method','random_seed','fitness','fitness_1','fitness_2','fitness_3',
                'poi_included','rating','tarif','total_duration','run_time']]
        df_doi = pd.concat([df_doi,df])
#     return df_doi
    df_doi.to_excel(filename,index=False)

In [115]:
random.seed(100)
doi_combi = [(np.round(random.uniform(0,1),2),np.round(random.uniform(0,1),2),np.round(random.uniform(0,1),2)) for i in range(10)]

In [116]:
doi_combi

[(0.15, 0.45, 0.77),
 (0.71, 0.73, 0.43),
 (0.8, 0.53, 0.08),
 (0.46, 0.05, 0.93),
 (0.95, 0.34, 0.31),
 (0.77, 0.2, 0.18),
 (0.19, 0.35, 0.63),
 (0.96, 0.21, 0.96),
 (0.56, 0.9, 0.82),
 (0.16, 0.65, 0.12)]

In [ ]:
doi_combi_analysis = []
for doi in doi_combi:
    print("doi combination = ",doi)
    random_ls = list(range(5,(5*50)+1,5))
    random_node_analysis ={
#         "acs_vrp": [],
#         "aco_vrp": [],
#         "bso_vrp": [],
#         "ga_vrp": [],
#         "ts_vrp": [],
#         "sa_vrp": [],
        "dka_vrp": [],
    }
    for i in range(len(random_ls)):
        # setting
        hotel,tur,_ = generate_data(n=30,random_state=random_ls[i]) #random state 50 blacklist
        timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
        start_day = 5 #start from saturday
        travel_days = 3
        dwaktu,dtarif,drating = doi

        print("random state = ",random_ls[i])

#         acs_vrp = ACS_VRP(random_state=100)
#         results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["acs_vrp"].append(results_acs_vrp)

#         aco_vrp = ACO_VRP(random_state=100)
#         results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["aco_vrp"].append(results_aco_vrp)

#         bso_vrp = BSO_VRP(random_state=100)
#         results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["bso_vrp"].append(results_bso_vrp)

#         ga_vrp = GA_VRP(random_state=100)
#         results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["ga_vrp"].append(results_ga_vrp)

#         ts_vrp = TS_VRP(random_state=100)
#         results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["ts_vrp"].append(results_ts_vrp)

#         sa_vrp = SA_VRP(random_state=100)
#         results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node_analysis["sa_vrp"].append(results_sa_vrp)

        dka_vrp = DKA_VRP_TD_2(random_state=100)
        results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day = start_day)
        random_node_analysis["dka_vrp"].append(results_dka_vrp)
    
    doi_combi_analysis.append(random_node_analysis)

doi combination =  (0.15, 0.45, 0.77)
141
random state =  5
141
random state =  10
125
random state =  15
115
random state =  20
109
random state =  25
149
random state =  30
127
random state =  35
127
random state =  40
162
random state =  45
174
random state =  50
162
random state =  55
154
random state =  60
172
random state =  65
143
random state =  70
115
random state =  75
114
random state =  80
139
random state =  85
157
random state =  90
136
random state =  95
174
random state =  100
103
random state =  105
109
random state =  110
172
random state =  115
139
random state =  120
162
random state =  125
127
random state =  130
122
random state =  135
115
random state =  140
185
random state =  145
100
random state =  150
103
random state =  155
172
random state =  160
155
random state =  165
114
random state =  170
115
random state =  175
136
random state =  180
128
random state =  185
136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  20

136
random state =  190
176
random state =  195
157
random state =  200
106
random state =  205
127
random state =  210
139
random state =  215
106
random state =  220
157
random state =  225
176
random state =  230
166
random state =  235
175
random state =  240
149
random state =  245
174
random state =  40
162
random state =  45
122
random state =  135
random state =  225
176
random state =  230
172
random state =  65
143
100
random state =  150
174
random state =  250
doi combination =  (0.16, 0.65, 0.12)
141
random state =  75
114
random state =  80
139
155
random state =  165
114


In [127]:
doi_combi_analysis

[{'dka_vrp': [{'solution': [[79, 14, 88, 45, 21, 44, 9, 33],
     [34, 55, 16, 53, 50, 10, 13],
     [11, 7, 6, 81, 18, 86, 36, 89, 1]],
    'fitness': 0.7525357998908905,
    'daily_fitness': [0.46737610321394224,
     0.45982400921260413,
     0.47302568893501223],
    'days_needed': 3,
    'poi_included': 24,
    'avg_rating': 4.641666666666666,
    'total_tarif': 54000,
    'total_duration': 123480,
    'run_time': 3.34188175201416},
   {'solution': [[3, 52, 44, 21, 86, 7, 46, 74, 22, 2],
     [71, 65, 66, 57, 32, 36],
     [11, 18, 88, 92, 33, 34, 94, 24, 13]],
    'fitness': 0.7771942267015056,
    'daily_fitness': [0.5539322550398497,
     0.4050452115190871,
     0.4644941197594039],
    'days_needed': 3,
    'poi_included': 25,
    'avg_rating': 4.596,
    'total_tarif': 25000,
    'total_duration': 127920,
    'run_time': 1.9629390239715576},
   {'solution': [[6, 68, 74, 60, 36, 18, 96, 24],
     [94, 10, 21, 16, 20, 2, 7, 92, 83],
     [39, 17, 79, 26, 32]],
    'fitness': 0

In [128]:
result_to_xlsx_rd_doi(doi_combi_analysis,doi_combi,random_ls,"dka_random_doi_combi_TD.xlsx")

## 87 POIs

In [130]:
def result_to_xlsx_all_poi(result,n,random_ls,filename):
    df_result = pd.DataFrame()
    for i in result:
        for j in range(len(result[i])):
            temp = pd.DataFrame(result[i][j],columns=["fitness","run_time"])
            temp['n'] = n[j]
            temp['method'] = i
            temp['random_seed'] = random_ls
            df_result = pd.concat([df_result,temp])
    df_result.to_excel(filename,index=False)

### All DOIs 1

In [131]:
n = [5,10,15,20,25,30,35,40,45,50,55,60,65]
run_time_analysis = {
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
max_runtime = 100*60 #100 minutes
for i in n:
    print("n = ",i)
    random_ls = list(range(5,(5*30)+1,5))
    random_node = {
#         "acs_vrp": [],
#         "aco_vrp": [],
#         "bso_vrp": [],
#         "ga_vrp": [],
#         "ts_vrp": [],
#         "sa_vrp": [],
        "dka_vrp": [],
    }
    
    for j in range(len(random_ls)):
        # setting
        hotel,tur,timematrix = generate_data(n=i,random_state=random_ls[j])
        timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
        start_day = 5 #start from saturday
        travel_days = 3
        dwaktu,dtarif,drating = 1,1,1
        
        print("random_state : ",random_ls[j])
        
#         acs_vrp = ACS_VRP(random_state=100)
#         results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["acs_vrp"].append([results_acs_vrp[key] for key in ["fitness","run_time"]])

#         aco_vrp = ACO_VRP(random_state=100)
#         results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["aco_vrp"].append([results_aco_vrp[key] for key in ["fitness","run_time"]])

#         bso_vrp = BSO_VRP(random_state=100)
#         results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["bso_vrp"].append([results_bso_vrp[key] for key in ["fitness","run_time"]])

#         ga_vrp = GA_VRP(random_state=100)
#         results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["ga_vrp"].append([results_ga_vrp[key] for key in ["fitness","run_time"]])

#         ts_vrp = TS_VRP(random_state=100)
#         results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["ts_vrp"].append([results_ts_vrp[key] for key in ["fitness","run_time"]])

#         sa_vrp = SA_VRP(random_state=100)
#         results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["sa_vrp"].append([results_sa_vrp[key] for key in ["fitness","run_time"]])

        dka_vrp = DKA_VRP_TD_2(random_state=100)
        results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
        random_node["dka_vrp"].append([results_dka_vrp[key] for key in ["fitness","run_time"]])
    
    for method in run_time_analysis:
        run_time_analysis[method].append(random_node[method])

n =  5
166
random_state :  5
141
random_state :  10
185
random_state :  15
100
random_state :  20
103
random_state :  25
136
random_state :  30
139
random_state :  35
157
random_state :  40
125
random_state :  45
166
random_state :  50
164
random_state :  55
172
random_state :  60
141
random_state :  65
127
random_state :  70
157
random_state :  75
147
random_state :  80
141
random_state :  85
172
random_state :  90
112
random_state :  95
139
random_state :  100
166
random_state :  105
155
random_state :  110
166
random_state :  115
127
random_state :  120
112
random_state :  125
100
random_state :  130
141
random_state :  135
164
random_state :  140
157
random_state :  145
172
random_state :  150
n =  10
143
random_state :  5
155
random_state :  10
157
random_state :  15
112
random_state :  20
103
random_state :  25
141
random_state :  30
164
random_state :  35
176
random_state :  40
100
random_state :  45
157
random_state :  50
164
random_state :  55
103
random_state :  60
185
random

172
random_state :  95
162
random_state :  100
166
random_state :  105
100
random_state :  110
146
random_state :  115
136
random_state :  120
155
random_state :  125
154
random_state :  130
112
random_state :  135
136
random_state :  140
166
random_state :  145
103
random_state :  150
n =  65
174
random_state :  5
103
random_state :  10
136
random_state :  15
185
random_state :  20
114
random_state :  25
146
random_state :  30
147
random_state :  35
109
random_state :  40
125
random_state :  45
125
random_state :  50
114
random_state :  55
106
random_state :  60
166
random_state :  65
175
random_state :  70
172
random_state :  75
162
random_state :  80
117
random_state :  85
149
random_state :  90
117
random_state :  95
136
random_state :  100
175
random_state :  105
143
random_state :  110
114
random_state :  115
166
random_state :  120
122
random_state :  125
139
random_state :  130
125
random_state :  135
154
random_state :  140
155
random_state :  145
174
random_state :  150
n =  

ValueError: Sample larger than population or is negative

In [132]:
run_time_analysis

{'dka_vrp': [[[0.7068672839506162, 0.08415675163269043],
   [0.90570987654321, 0.07994937896728516],
   [0.8888734567901236, 0.07904505729675293],
   [0.8420061728395062, 0.07326555252075195],
   [0.692361111111111, 0.06784319877624512],
   [0.7064043209876539, 0.07936358451843262],
   [0.7016975308641976, 0.07026791572570801],
   [0.7403703703703705, 0.054775238037109375],
   [0.7509259259259263, 0.07697415351867676],
   [0.6996141975308644, 0.06736040115356445],
   [0.7325617283950615, 0.1115102767944336],
   [0.7117283950617285, 0.13657689094543457],
   [0.7147376543209879, 0.07949137687683105],
   [0.7331018518518521, 0.08100748062133789],
   [0.7320987654320987, 0.07653188705444336],
   [0.6526234567901236, 0.10779285430908203],
   [0.690046296296296, 0.07385683059692383],
   [0.6897376543209877, 0.06328964233398438],
   [0.7132716049382716, 0.07996845245361328],
   [0.6996913580246913, 0.1656324863433838],
   [0.7047067901234568, 0.06659674644470215],
   [0.7123456790123456, 0.09

In [133]:
result_to_xlsx_all_poi(run_time_analysis,n,random_ls,"dka_various_numbers_pois_all_1_TD.xlsx")

In [135]:
#continue
n = [5,10,15,20,25,30,35,40,45,50,55,60,65]
run_time_analysis = {
#     "acs_vrp": [],
#     "aco_vrp": [],
#     "bso_vrp": [],
#     "ga_vrp": [],
#     "ts_vrp": [],
#     "sa_vrp": [],
    "dka_vrp": [],
}
max_runtime = 100*60 #100 minutes
for i in n:
    print("n = ",i)
    random_ls = list(range(155,(5*50)+1,5))
    random_node = {
#         "acs_vrp": [],
#         "aco_vrp": [],
#         "bso_vrp": [],
#         "ga_vrp": [],
#         "ts_vrp": [],
#         "sa_vrp": [],
        "dka_vrp": [],
    }
    
    for j in range(len(random_ls)):
        # setting
        hotel,tur,timematrix = generate_data(n=i,random_state=random_ls[j])
        timematrix = build_travel_time_dict(weekday_file, weekend_file, hotel, tur)
        start_day = 5 #start from saturday
        travel_days = 3
        dwaktu,dtarif,drating = 1,1,1
        
        print("random_state : ",random_ls[j])
        
#         acs_vrp = ACS_VRP(random_state=100)
#         results_acs_vrp = get_analysis(acs_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["acs_vrp"].append([results_acs_vrp[key] for key in ["fitness","run_time"]])

#         aco_vrp = ACO_VRP(random_state=100)
#         results_aco_vrp = get_analysis(aco_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["aco_vrp"].append([results_aco_vrp[key] for key in ["fitness","run_time"]])

#         bso_vrp = BSO_VRP(random_state=100)
#         results_bso_vrp = get_analysis(bso_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["bso_vrp"].append([results_bso_vrp[key] for key in ["fitness","run_time"]])

#         ga_vrp = GA_VRP(random_state=100)
#         results_ga_vrp = get_analysis(ga_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["ga_vrp"].append([results_ga_vrp[key] for key in ["fitness","run_time"]])

#         ts_vrp = TS_VRP(random_state=100)
#         results_ts_vrp = get_analysis(ts_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["ts_vrp"].append([results_ts_vrp[key] for key in ["fitness","run_time"]])

#         sa_vrp = SA_VRP(random_state=100)
#         results_sa_vrp = get_analysis(sa_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating)
#         random_node["sa_vrp"].append([results_sa_vrp[key] for key in ["fitness","run_time"]])

        dka_vrp = DKA_VRP_TD_2(random_state=100)
        results_dka_vrp = get_analysis(dka_vrp,hotel,tur,timematrix,travel_days,dwaktu,dtarif,drating,start_day=start_day)
        random_node["dka_vrp"].append([results_dka_vrp[key] for key in ["fitness","run_time"]])
    
    for method in run_time_analysis:
        run_time_analysis[method].append(random_node[method])

n =  5
172
random_state :  155
185
random_state :  160
117
random_state :  165
122
random_state :  170
122
random_state :  175
149
random_state :  180
164
random_state :  185
117
random_state :  190
155
random_state :  195
185
random_state :  200
141
random_state :  205
155
random_state :  210
106
random_state :  215
139
random_state :  220
176
random_state :  225
166
random_state :  230
106
random_state :  235
155
random_state :  240
127
random_state :  245
147
random_state :  250
n =  10
149
random_state :  155
147
random_state :  160
176
random_state :  165
172
random_state :  170
146
random_state :  175
109
random_state :  180
149
random_state :  185
106
random_state :  190
154
random_state :  195
162
random_state :  200
109
random_state :  205
109
random_state :  210
125
random_state :  215
117
random_state :  220
172
random_state :  225
143
random_state :  230
103
random_state :  235
166
random_state :  240
141
random_state :  245
185
random_state :  250
n =  15
114
random_state 

In [136]:
run_time_analysis

{'dka_vrp': [[[0.7309104938271606, 0.05206418037414551],
   [0.6607253086419753, 0.11035704612731934],
   [0.6940586419753086, 0.06307029724121094],
   [0.7090277777777775, 0.09086728096008301],
   [0.6996141975308637, 0.0725245475769043],
   [0.8685956790123458, 0.08214569091796875],
   [0.7350308641975308, 0.08509421348571777],
   [0.6787037037037041, 0.12264728546142578],
   [0.6964506172839506, 0.08600234985351562],
   [0.674305555555556, 0.06009554862976074],
   [0.7050000000000004, 0.06067609786987305],
   [0.6773919753086418, 0.047310829162597656],
   [0.7117283950617285, 0.06264805793762207],
   [0.711574074074074, 0.12050652503967285],
   [0.6730709876543214, 0.07327795028686523],
   [0.7196913580246916, 0.07466626167297363],
   [0.6992283950617294, 0.06974554061889648],
   [0.7060956790123455, 0.07434344291687012],
   [0.7206018518518517, 0.08266830444335938],
   [0.703703703703704, 0.09763789176940918]],
  [[0.6784104938271605, 0.28208279609680176],
   [0.6291512345679015, 0

In [137]:
result_to_xlsx_all_poi(run_time_analysis,n,random_ls,"dka_various_numbers_pois_all_1_v2_TD.xlsx")

# Uji Statistik

In [5]:
def uji_statistik(first_data,second_data,alternative="two-sided",first_data_name="first",second_data_name="second"):
    # first_data is the target data, e.g. the results of Hybrid ACS-BSO 
    # second_data is a list of results from other method
    # the test is using Wilcoxon Rank-Sum Test
    p_values = []
    second_data_name = ["second" for i in range(len(second_data))] if second_data_name == "second" else second_data_name
    for i in range(len(second_data)):
        statistic, p_value = stats.ranksums(first_data, second_data[i],alternative=alternative)
        p_values.append(p_value)
        print(f"Wilcoxon Rank-Sum Test ({first_data_name} vs. {second_data_name[i]}):")
        print("Test Statistic:", statistic)
        print("p-value:", p_value)
        print("")
    return p_values

def save_to_excel_sheet(df,filename,sheet_name):
    try:
        with pd.ExcelWriter(filename, mode="a", engine="openpyxl", if_sheet_exists='replace') as writer:

            # use to_excel function and specify the sheet_name and index to 
            # store the dataframe in specified sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except:
        with pd.ExcelWriter(filename, mode="w", engine="openpyxl") as writer:

            # use to_excel function and specify the sheet_name and index to 
            # store the dataframe in specified sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)

## Hyperparameter Tuning

### Iteration Convergence

In [52]:
results_df = pd.read_excel("dka_iteration_convergence.xlsx")
results_df.head()

,iter,best_fitness,doi_combi
0,1,0.596798,"1,1,1"
1,2,0.596798,"1,1,1"
2,3,0.598599,"1,1,1"
3,4,0.598599,"1,1,1"
4,5,0.598599,"1,1,1"


In [53]:
results_df = results_df.pivot(index=['iter'],columns='doi_combi',values='best_fitness').reset_index()
results_df

doi_combi,iter,"0,0,1","0,1,0","1,0,0","1,1,1"
0,1,0.735714,0.754738,0.604745,0.596798
1,2,0.735714,0.754738,0.604745,0.596798
2,3,0.735714,0.754738,0.606019,0.598599
3,4,0.737362,0.754738,0.606019,0.598599
4,5,0.737362,0.754738,0.606019,0.598599
...,...,...,...,...,...
116,117,NaN,NaN,NaN,0.643400
117,118,NaN,NaN,NaN,0.643400
118,119,NaN,NaN,NaN,0.643400
119,120,NaN,NaN,NaN,0.643400


In [54]:
save_to_excel_sheet(results_df,filename="dka_experiment_results.xlsx",sheet_name="iteration_convergence_results")

### Parameter Tuning

In [25]:
results_df = pd.read_excel("dka_tuning_n_parameter.xlsx")
results_df.head()

,n,fitness,run_time
0,5,0.643400,2.266844
1,10,0.666665,6.990387
2,15,0.664651,17.575596
3,20,0.673871,60.898961


In [26]:
save_to_excel_sheet(results_df,filename="dka_experiment_results.xlsx",sheet_name="tuning_n_parameter_results")

## Outlier Sensitivity

In [6]:
results_df = pd.read_excel("dka_outlier_sensitivity.xlsx")
results_df2 = pd.read_excel("kma_outlier_sensitivity.xlsx")
results_df3 = pd.read_excel("pso_outlier_sensitivity.xlsx")
results_df4 = pd.read_excel("ils_outlier_sensitivity.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,methods,num_outlier,travel_days
0,dka_vrp,0,1
1,acs_vrp,0,1
2,aco_vrp,0,1
3,bso_vrp,0,1
4,ga_vrp,0,1


In [7]:
results_df = results_df.pivot(index=['travel_days'],columns='methods',values='num_outlier').reset_index()

In [8]:
save_to_excel_sheet(results_df,filename="dka_experiment_results.xlsx",sheet_name="outlier_sensitivity_results")

## General performance

### All DOI 1

In [9]:
results_df = pd.read_excel("dka_random_nodes_alldoi_1.xlsx")
results_df2 = pd.read_excel("kma_random_nodes_alldoi_1.xlsx")
results_df3 = pd.read_excel("pso_random_nodes_alldoi_1.xlsx")
results_df4 = pd.read_excel("ils_random_nodes_alldoi_1.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,random_seed,fitness,fitness_1,fitness_2,fitness_3,poi_included,rating,tarif,total_duration,run_time,method
0,5,0.681458,0.709375,0.625561,0.650876,21,4.614286,37000,125100,4.769661,acs_vrp
1,10,0.644172,0.631843,0.602275,0.602765,20,4.580000,24500,124620,3.170985,acs_vrp
2,15,0.611977,0.603838,0.579350,0.546002,21,4.633333,81500,126480,2.801009,acs_vrp
3,20,0.678256,0.665319,0.629272,0.618132,22,4.590909,41000,122520,3.208303,acs_vrp
4,25,0.628851,0.631071,0.607775,0.587981,20,4.560000,86000,121860,5.879171,acs_vrp


In [10]:
grouped = results_df.groupby('method').agg(fitness = ('fitness','mean'),
                                           std_fitness = ('fitness','std'),
                                           run_time = ('run_time','mean'),
                                           std_run_time = ('run_time','std')).reset_index()
grouped.sort_values('fitness',ascending=False)

,method,fitness,std_fitness,run_time,std_run_time
1,acs_vrp,0.659326,0.024286,3.746193,1.383804
12,sa_vrp,0.658175,0.024231,0.971710,0.035800
9,kma_vrp,0.656306,0.024101,5.064276,2.320104
10,mga_vrp,0.655660,0.027014,0.540331,0.072410
11,pso_vrp,0.647372,0.025445,0.604844,0.156425
14,vns_vrp,0.642761,0.028333,0.167079,0.074258
3,dka_vrp,0.640759,0.023786,0.462857,0.215773
15,woa_vrp,0.640459,0.024909,0.414673,0.132829
13,ts_vrp,0.640236,0.025841,0.665049,0.023877
0,aco_vrp,0.639912,0.027561,2.691196,0.641916


In [11]:
metrics = ["fitness","run_time"]

df_stat = pd.DataFrame()
for metric in metrics:
    print("metric: ",metric)
    metrics = metric
    alternative = "two-sided"
    first_data = results_df[results_df['method']=="dka_vrp"][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[results_df['method']==method][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

metric:  fitness
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -3.660618353867727
p-value: 0.0002516072978259805

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -0.40673537265196963
p-value: 0.6842023576175498

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 8.293265310174906
p-value: 1.1018121083880995e-16

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 1.8751190061243346
p-value: 0.060776353380831025

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -0.42052301240288387
p-value: 0.6741034218588211

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -3.433122297977642
p-value: 0.0005966728938228825

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -3.260776801091214
p-value: 0.0011110746216913378

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -0.2137084161391705
p-value: 0.8307744478348679

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: -1.0133915216921956
p-val

In [12]:
df_stat

,metrics,method,p_value
0,fitness,acs_vrp,2.516073e-04
1,fitness,aco_vrp,6.842024e-01
2,fitness,bso_vrp,1.101812e-16
3,fitness,ga_vrp,6.077635e-02
4,fitness,ts_vrp,6.741034e-01
5,fitness,sa_vrp,5.966729e-04
6,fitness,kma_vrp,1.111075e-03
7,fitness,woa_vrp,8.307744e-01
8,fitness,vns_vrp,3.108732e-01
9,fitness,pso_vrp,1.082168e-01


In [13]:
save_to_excel_sheet(grouped.sort_values('fitness',ascending=False),filename="dka_experiment_results.xlsx",sheet_name="alldoi_1_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="alldoi_1_p_value")

### Travel Duration 1

In [14]:
results_df = pd.read_excel("dka_random_nodes_dwaktu_1.xlsx")
results_df2 = pd.read_excel("kma_random_nodes_dwaktu_1.xlsx")
results_df3 = pd.read_excel("pso_random_nodes_dwaktu_1.xlsx")
results_df4 = pd.read_excel("ils_random_nodes_dwaktu_1.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,random_seed,fitness,fitness_1,fitness_2,fitness_3,poi_included,rating,tarif,total_duration,run_time,method
0,5,0.648032,0.510880,0.424074,0.477778,23,4.556522,104500,121980,3.011678,acs_vrp
1,10,0.642361,0.487963,0.472685,0.440741,23,4.534783,196000,124920,4.785303,acs_vrp
2,15,0.631211,0.481481,0.438426,0.463657,23,4.604348,260500,129540,3.239691,acs_vrp
3,20,0.663850,0.483102,0.487037,0.457639,24,4.558333,283000,122400,3.555043,acs_vrp
4,25,0.626852,0.468287,0.459491,0.459259,22,4.500000,205500,124320,7.371741,acs_vrp


In [15]:
grouped = results_df.groupby('method').agg(fitness = ('fitness','mean'),
                                           std_fitness = ('fitness','std'),
                                           run_time = ('run_time','mean'),
                                           std_run_time = ('run_time','std')).reset_index()
grouped.sort_values('fitness',ascending=False)

,method,fitness,std_fitness,run_time,std_run_time
1,acs_vrp,0.653629,0.019778,3.864807,1.499588
9,kma_vrp,0.652954,0.020802,6.356812,2.741273
0,aco_vrp,0.652414,0.022920,2.674487,0.692401
12,sa_vrp,0.651903,0.019467,0.946983,0.011019
10,mga_vrp,0.646004,0.021900,0.513543,0.082797
11,pso_vrp,0.643566,0.019813,0.620332,0.201708
13,ts_vrp,0.641319,0.022155,0.648231,0.017646
14,vns_vrp,0.640123,0.022411,0.164907,0.069094
3,dka_vrp,0.637856,0.023063,0.436884,0.205289
15,woa_vrp,0.634821,0.022032,0.430723,0.106478


In [16]:
metrics = ["fitness","run_time"]

df_stat = pd.DataFrame()
for metric in metrics:
    print("metric: ",metric)
    metrics = metric
    alternative = "two-sided"
    first_data = results_df[results_df['method']=="dka_vrp"][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[results_df['method']==method][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

metric:  fitness
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -3.4917197669190276
p-value: 0.0004799214751372957

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -3.102218943955701
p-value: 0.0019207584104581645

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 6.587044890999271
p-value: 4.486666614971876e-11

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 1.9233757452525344
p-value: 0.054432878152186996

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -0.8582805744944105
p-value: 0.390737553235738

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -3.029833835263401
p-value: 0.0024468831891312623

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -3.2228607917762004
p-value: 0.001269172182826938

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.7548732763625539
p-value: 0.4503250216950171

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: -0.6445721583552401
p-value

In [17]:
save_to_excel_sheet(grouped.sort_values('fitness',ascending=False),filename="dka_experiment_results.xlsx",sheet_name="dwaktu_1_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="dwaktu_1_p_value")

### Tarif 1

In [20]:
results_df = pd.read_excel("dka_random_nodes_dtarif_1.xlsx")
results_df2 = pd.read_excel("kma_random_nodes_dtarif_1.xlsx")
results_df3 = pd.read_excel("pso_random_nodes_dtarif_1.xlsx")
results_df4 = pd.read_excel("ils_random_nodes_dtarif_1.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,random_seed,fitness,fitness_1,fitness_2,fitness_3,poi_included,rating,tarif,total_duration,run_time,method
0,5,0.809434,0.593082,0.592138,0.583648,21,4.604762,43000,128340,1.987593,acs_vrp
1,10,0.807227,0.575221,0.607817,0.598083,20,4.565000,29500,127380,4.700382,acs_vrp
2,15,0.791200,0.622184,0.550649,0.561034,21,4.623810,83500,129600,3.271467,acs_vrp
3,20,0.841992,0.639924,0.627036,0.550588,22,4.586364,39000,127800,5.548777,acs_vrp
4,25,0.780875,0.588576,0.555925,0.569949,21,4.528571,106000,126180,3.074423,acs_vrp


In [21]:
grouped = results_df.groupby('method').agg(fitness = ('fitness','mean'),
                                           std_fitness = ('fitness','std'),
                                           run_time = ('run_time','mean'),
                                           std_run_time = ('run_time','std')).reset_index()
grouped.sort_values('fitness',ascending=False)

,method,fitness,std_fitness,run_time,std_run_time
9,kma_vrp,0.826179,0.021846,4.624728,1.966471
12,sa_vrp,0.825749,0.021078,0.957705,0.016814
1,acs_vrp,0.825103,0.022411,3.643509,1.205659
11,pso_vrp,0.814867,0.021476,0.515292,0.151636
10,mga_vrp,0.808954,0.026028,0.353386,0.082856
13,ts_vrp,0.807096,0.025027,0.663548,0.018284
15,woa_vrp,0.803911,0.026580,0.365912,0.086163
3,dka_vrp,0.799982,0.026349,0.406919,0.190630
14,vns_vrp,0.795416,0.034217,0.117696,0.053228
4,ga_vrp,0.794564,0.027925,0.767411,0.348168


In [22]:
metrics = ["fitness","run_time"]

df_stat = pd.DataFrame()
for metric in metrics:
    print("metric: ",metric)
    metrics = metric
    alternative = "two-sided"
    first_data = results_df[results_df['method']=="dka_vrp"][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[results_df['method']==method][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

metric:  fitness
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -4.5154520184244085
p-value: 6.318185160126762e-06

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: 1.9923139440071056
p-value: 0.046336622708105214

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 8.59659338469502
p-value: 8.211730646555817e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 0.8272583850548535
p-value: 0.40809062551680486

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -1.2684628570841088
p-value: 0.2046327068387458

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -4.639540776182637
p-value: 3.4918423568998417e-06

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -4.5774963973035225
p-value: 4.7057389178179324e-06

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -0.8065769254284822
p-value: 0.41991027219796906

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 0.5928685092893117
p-val

In [23]:
save_to_excel_sheet(grouped.sort_values('fitness',ascending=False),filename="dka_experiment_results.xlsx",sheet_name="dtarif_1_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="dtarif_1_p_value")

### Rating 1

In [24]:
results_df = pd.read_excel("dka_random_nodes_drating_1.xlsx")
results_df2 = pd.read_excel("kma_random_nodes_drating_1.xlsx")
results_df3 = pd.read_excel("pso_random_nodes_drating_1.xlsx")
results_df4 = pd.read_excel("ils_random_nodes_drating_1.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,random_seed,fitness,fitness_1,fitness_2,fitness_3,poi_included,rating,tarif,total_duration,run_time,method
0,5,0.826339,0.594444,0.498148,0.445833,23,4.595652,179500,128040,2.198029,acs_vrp
1,10,0.774953,0.477083,0.402381,0.358333,23,4.569565,142000,129420,5.177960,acs_vrp
2,15,0.738227,0.454762,0.320748,0.310544,22,4.640909,151500,126180,3.272198,acs_vrp
3,20,0.795721,0.550000,0.427778,0.377778,23,4.591304,235000,127980,1.410371,acs_vrp
4,25,0.750396,0.483333,0.395455,0.365152,21,4.566667,139000,127080,2.331297,acs_vrp


In [25]:
grouped = results_df.groupby('method').agg(fitness = ('fitness','mean'),
                                           std_fitness = ('fitness','std'),
                                           run_time = ('run_time','mean'),
                                           std_run_time = ('run_time','std')).reset_index()
grouped.sort_values('fitness',ascending=False)

,method,fitness,std_fitness,run_time,std_run_time
9,kma_vrp,0.799579,0.033538,5.528577,2.419058
1,acs_vrp,0.796156,0.035199,3.169225,1.399642
12,sa_vrp,0.793384,0.033063,0.963424,0.018638
11,pso_vrp,0.787471,0.034832,0.540462,0.162198
13,ts_vrp,0.785960,0.033824,0.673597,0.021443
15,woa_vrp,0.782584,0.033836,0.398832,0.088496
3,dka_vrp,0.780811,0.031814,0.407809,0.196857
14,vns_vrp,0.779596,0.035539,0.123323,0.056490
10,mga_vrp,0.779486,0.032997,0.339333,0.081625
0,aco_vrp,0.775737,0.034899,2.412886,0.363135


In [26]:
metrics = ["fitness","run_time"]

df_stat = pd.DataFrame()
for metric in metrics:
    print("metric: ",metric)
    metrics = metric
    alternative = "two-sided"
    first_data = results_df[results_df['method']=="dka_vrp"][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[results_df['method']==method][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

metric:  fitness
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -2.323217298029047
p-value: 0.02016748197034859

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: 0.6824881676702542
p-value: 0.49493032211762333

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 6.845563136328913
p-value: 7.617561341263654e-12

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 0.7686609161134681
p-value: 0.44209463261794313

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -0.854833664556682
p-value: 0.3926432384291334

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -1.95784484462982
p-value: 0.050248221503831035

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -2.7851032296846734
p-value: 0.00535106842508092

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -0.272305885080556
p-value: 0.7853868315060774

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 0.19302695651279916
p-value: 0.84693

In [27]:
save_to_excel_sheet(grouped.sort_values('fitness',ascending=False),filename="dka_experiment_results.xlsx",sheet_name="drating_1_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="drating_1_p_value")

## DOI Combination

In [29]:
results_df = pd.read_excel("dka_random_doi_combi.xlsx")
results_df2 = pd.read_excel("kma_random_doi_combi.xlsx")
results_df3 = pd.read_excel("pso_random_doi_combi.xlsx")
results_df4 = pd.read_excel("ils_random_doi_combi.xlsx")
results_df = pd.concat([results_df,results_df2,results_df3,results_df4])
results_df.head()

,dwaktu,dtarif,drating,method,random_seed,fitness,fitness_1,fitness_2,fitness_3,poi_included,rating,tarif,total_duration,run_time
0,0.15,0.45,0.77,acs_vrp,5,0.788916,0.554363,0.536150,0.613935,23,4.591304,75000,128280,5.805615
1,0.15,0.45,0.77,acs_vrp,10,0.736776,0.571812,0.449716,0.491406,21,4.580952,52000,125100,4.605406
2,0.15,0.45,0.77,acs_vrp,15,0.714983,0.475079,0.449430,0.475128,22,4.627273,110500,133320,3.156808
3,0.15,0.45,0.77,acs_vrp,20,0.783644,0.601456,0.538273,0.512053,23,4.595652,57000,129420,4.294260
4,0.15,0.45,0.77,acs_vrp,25,0.730412,0.547887,0.491160,0.474618,22,4.540909,146000,131040,4.416386


In [30]:
grouped = results_df.groupby(['dwaktu','dtarif','drating','method']).agg(fitness = ('fitness','mean'),
                                                                         std_fitness = ('fitness','std'),
                                                                         run_time = ('run_time','mean'),
                                                                         std_run_time = ('run_time','std'))

grouped

fitness  std_fitness  run_time  std_run_time
dwaktu dtarif drating method                                                
0.15   0.45   0.77    aco_vrp  0.746655     0.029569  3.153367      0.906980
                      acs_vrp  0.763987     0.025995  4.320484      1.576167
                      bso_vrp  0.672562     0.041997  0.401293      0.162018
                      dka_vrp  0.751793     0.029377  0.497686      0.259498
                      ga_vrp   0.741615     0.028391  0.738393      0.331004
...                                 ...          ...       ...           ...
0.96   0.21   0.96    pso_vrp  0.635791     0.025156  0.663700      0.200946
                      sa_vrp   0.643477     0.025331  0.942659      0.020909
                      ts_vrp   0.631328     0.025686  0.653747      0.019269
                      vns_vrp  0.632843     0.031956  0.170804      0.074414
                      woa_vrp  0.628678     0.027059  0.443154      0.119178

[160 rows x 4 columns]

In [31]:
random.seed(100)
doi_combi = [(np.round(random.uniform(0,1),2),np.round(random.uniform(0,1),2),np.round(random.uniform(0,1),2)) for i in range(10)]

In [32]:
df_stat = pd.DataFrame()
for doi in doi_combi:
    print("doi combination : ",doi)
    metrics = "fitness"
    print("metric: ",metrics)
    alternative = "two-sided"
    first_data = results_df[(results_df['method']=="dka_vrp")&
                        (results_df['dwaktu']==doi[0])&
                        (results_df['dtarif']==doi[1])&
                        (results_df['drating']==doi[2])][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[(results_df['method']==method)&
                        (results_df['dwaktu']==doi[0])&
                        (results_df['dtarif']==doi[1])&
                        (results_df['drating']==doi[2])][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'dwaktu':doi[0],
                                               'dtarif':doi[1],
                                               'drating':doi[2],
                                               'metrics':metrics,
                                               'method':second_data_name,
                                               'p_value':p_values})])

doi combination :  (0.15, 0.45, 0.77)
metric:  fitness
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -2.143977981267162
p-value: 0.03203465109154384

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: 1.3029319564613944
p-value: 0.19259799276641898

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 7.486688384746425
p-value: 7.063313496311528e-14

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 2.0957212421389624
p-value: 0.03610692636685788

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: 0.3446909937728556
p-value: 0.7303266896009318

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -1.895800465750706
p-value: 0.057986434276740335

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -2.130190341516248
p-value: 0.03315590213557952

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.97892242231491
p-value: 0.3276183123071674

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -2.8954043476919873
p-value: 0.003786704151057738

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.33779717389739855
p-value: 0.7355160397405875

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: -0.7307449067984539
p-value: 0.464934980549941

Wilcoxon Rank-Sum Test (dka_vrp vs. pso_vrp):
Test Statistic: -0.9582409626885386
p-value: 0.33794126596326657

Wilcoxon Rank-Sum Test (dka_vrp vs. ils_vrp):
Test Statistic: 3.8260700308786975
p-value: 0.00013020526280546052

Wilcoxon Rank-Sum Test (dka_vrp vs. ils_sa_vrp):
Test Statistic: 2.1508718011426193
p-value: 0.031486319835581415

Wilcoxon Rank-Sum Test (dka_vrp vs. ils_ts_vrp):
Test Statistic: 3.9019020495087258
p-value: 9.543977939453069e-05

Wilcoxon Rank-Sum Test (dka_vrp vs. ils_sa_ts_vrp):
Test Statistic: 2.0957212421389624
p-value: 0.03610692636685788

Wilcoxon Rank-Sum Test (dka_vrp vs. mga_vrp):
Test Statistic: -2.6403330123000743
p-value: 0

In [33]:
df_stat2 = pd.DataFrame()
for doi in doi_combi:
    print("doi combination : ",doi)
    metrics = "run_time"
    print("metric: ",metrics)
    alternative = "two-sided"
    first_data = results_df[(results_df['method']=="dka_vrp")&
                        (results_df['dwaktu']==doi[0])&
                        (results_df['dtarif']==doi[1])&
                        (results_df['drating']==doi[2])][metrics].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[~results_df['method'].isin(["dka_vrp"])]['method'].unique():
        second_data.append(results_df[(results_df['method']==method)&
                        (results_df['dwaktu']==doi[0])&
                        (results_df['dtarif']==doi[1])&
                        (results_df['drating']==doi[2])][metrics].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat2 = pd.concat([df_stat2,pd.DataFrame({'dwaktu':doi[0],
                                               'dtarif':doi[1],
                                               'drating':doi[2],
                                               'metrics':metrics,
                                               'method':second_data_name,
                                               'p_value':p_values})])

doi combination :  (0.15, 0.45, 0.77)
metric:  run_time
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 1.3856577949668796
p-value: 0.16585142133848707

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: -4.2948497824097815
p-value: 1.7481166264969505e-05

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -4.329318881787067
p-value: 1.4957122399174373e-05

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -7.376387266739111
p-value: 1.62642478199907e-13

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.46533284159335514
p-value: 0.6416931196450413

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):

Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 1.7234549688642782
p-value: 0.08480628179936625

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: -4.680903695435379
p-value: 2.856130942745735e-06

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -5.177258726468292
p-value: 2.2516985645008115e-07

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -8.28637149029945
p-value: 1.1675572001380731e-16

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -0.5170364906592835
p-value: 0.6051306860284573

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 7.948574316402051
p-value: 1.88670144807

In [34]:
df_stat = df_stat.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='p_value').reset_index()
df_stat2 = df_stat2.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='p_value').reset_index()
df_stat = pd.concat([df_stat,df_stat2])

In [35]:
grouped1 = grouped.reset_index().copy()
grouped1['metrics'] = 'fitness'
grouped1 = grouped1.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='fitness').reset_index()

grouped2 = grouped.reset_index().copy()
grouped2['metrics'] = 'run_time'
grouped2 = grouped2.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='run_time').reset_index()

grouped3 = grouped.reset_index().copy()
grouped3['metrics'] = 'std_fitness'
grouped3 = grouped3.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='std_fitness').reset_index()

grouped4 = grouped.reset_index().copy()
grouped4['metrics'] = 'std_run_time'
grouped4 = grouped4.pivot(index=['metrics','dwaktu','dtarif','drating'],columns='method',values='std_run_time').reset_index()


grouped = pd.concat([grouped1,grouped2,grouped3,grouped4])

In [36]:
save_to_excel_sheet(grouped,filename="dka_experiment_results.xlsx",sheet_name="doi_combi_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="doi_combi_p_value")

## All POI

In [37]:
results_df1 = pd.read_excel("dka_various_numbers_pois_all_1.xlsx")
results_df2 = pd.read_excel("dka_various_numbers_pois_all_1_v2.xlsx")
results_df3 = pd.read_excel("kma_various_numbers_pois_all_1.xlsx")
results_df4 = pd.read_excel("pso_various_numbers_pois_all_1.xlsx")
results_df5 = pd.read_excel("ils_various_numbers_pois_all_1.xlsx")
results_df = pd.concat([results_df1,results_df2,results_df3,results_df4,results_df5])
results_df = results_df.reset_index(drop=True)
results_df.head()

,fitness,run_time,n,method,random_seed
0,0.704398,0.170605,5,acs_vrp,5
1,0.710309,0.173731,5,acs_vrp,10
2,0.704784,0.134615,5,acs_vrp,15
3,0.719136,0.110273,5,acs_vrp,20
4,0.715586,0.118830,5,acs_vrp,25


In [38]:
grouped = results_df.groupby(['method','n']).agg(fitness = ('fitness','mean'),
                                                 std_fitness = ('fitness','std'),
                                                 run_time = ('run_time','mean'),
                                                 std_run_time = ('run_time','std')).copy()

In [39]:
n = results_df['n'].unique()
n.sort()
metric = "fitness"
alternative = "two-sided"

df_stat = pd.DataFrame()
for i in n:
    print("n: ",i)
    first_data = results_df[(results_df['method']=="dka_vrp")&(results_df['n']==i)][metric].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[results_df['method'] != "dka_vrp"]['method'].unique():
        second_data.append(results_df[(results_df['method']==method)&(results_df['n']==i)][metric].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat = pd.concat([df_stat,pd.DataFrame({'n':i,
                                               'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

n:  5
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: 0.017234549688642783
p-value: 0.9862494996122527

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: 0.2309429658278133
p-value: 0.8173591109195852

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 0.6514659782306972
p-value: 0.5147457320739065

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 0.04136291925274268
p-value: 0.9670065736331777

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: 0.0
p-value: 1.0

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: 0.15511094719778504
p-value: 0.8767338520233376

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: 0.0
p-value: 1.0

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.04825673912819979
p-value: 0.9615116315341786

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 0.2309429658278133
p-value: 0.8173591109195852

Wilcoxon Rank-Sum Test (dka_vrp vs. pso_vrp):
Test Statistic: 

Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -3.660618353867727
p-value: 0.0002516072978259805

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -0.40673537265196963
p-value: 0.6842023576175498

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 8.293265310174906
p-value: 1.1018121083880995e-16

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 1.8751190061243346
p-value: 0.060776353380831025

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -0.42052301240288387
p-value: 0.6741034218588211

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -3.433122297977642
p-value: 0.0005966728938228825

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -3.260776801091214
p-value: 0.0011110746216913378

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -0.2137084161391705
p-value: 0.8307744478348679

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: -1.0133915216921956
p-value: 0.31087318745

Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.320840589676735
p-value: 8.734165225620056e-17

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -7.693502981010138
p-value: 1.4316037008407338e-14

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 7.272979968607254
p-value: 3.5164248329970405e-13

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: 0.8479398446812249
p-value: 0.3964714729625185

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -4.398257080541638
p-value: 1.0912367447666874e-05

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -6.045880030775888
p-value: 1.4859664116044336e-09

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.010618695281165
p-value: 1.141328016223929e-15

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -3.0194931054502154
p-value: 0.0025319805586937762

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: -3.109112763831158
p-value: 0.001876

In [40]:
n = results_df['n'].unique()
n.sort()
metric = "run_time"
alternative = "two-sided"

df_stat2 = pd.DataFrame()
for i in n:
    print("n: ",i)
    first_data = results_df[(results_df['method']=="dka_vrp")&(results_df['n']==i)][metric].values
    first_data_name = "dka_vrp"

    second_data = []
    second_data_name = []
    for method in results_df[results_df['method'] != "dka_vrp"]['method'].unique():
        second_data.append(results_df[(results_df['method']==method)&(results_df['n']==i)][metric].values)
        second_data_name.append(method)

    p_values = uji_statistik(first_data,second_data,alternative=alternative,first_data_name=first_data_name,second_data_name=second_data_name)
    df_stat2 = pd.concat([df_stat2,pd.DataFrame({'n':i,
                                               'metrics':metric,
                                               'method':second_data_name,
                                               'p_value':p_values})])

n:  5
Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 5.983835651896774
p-value: 2.1794322776441375e-09

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: -8.59659338469502
p-value: 8.211730646555817e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 8.617274844321392
p-value: 6.856

Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: 1.8199684471206778
p-value: 0.06876381010880288

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: -5.294453664351063
p-value: 1.1937275114504229e-07

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -5.708082856878489
p-value: 1.1425580001484516e-08

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -8.300159130050364
p-value: 1.0397203684846393e-16

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 0.9513471428130815
p-value: 0.34142818298452027

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 7.73486590026288
p-value: 1.0351215406

Wilcoxon Rank-Sum Test (dka_vrp vs. acs_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. aco_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. bso_vrp):
Test Statistic: -8.603487204570477
p-value: 7.73298779987613e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. ga_vrp):
Test Statistic: -2.5782886334209603
p-value: 0.009929102314560907

Wilcoxon Rank-Sum Test (dka_vrp vs. ts_vrp):
Test Statistic: -3.2814582607175855
p-value: 0.0010327180791226923

Wilcoxon Rank-Sum Test (dka_vrp vs. sa_vrp):
Test Statistic: -7.659033881632852
p-value: 1.873369754505005e-14

Wilcoxon Rank-Sum Test (dka_vrp vs. kma_vrp):
Test Statistic: -8.617274844321392
p-value: 6.856641447475644e-18

Wilcoxon Rank-Sum Test (dka_vrp vs. woa_vrp):
Test Statistic: 1.7234549688642782
p-value: 0.08480628179936625

Wilcoxon Rank-Sum Test (dka_vrp vs. vns_vrp):
Test Statistic: 7.852060838145651
p-value: 4.0925586665

In [41]:
df_stat = df_stat.pivot(index=['metrics','n'],columns='method',values='p_value').reset_index()
df_stat2 = df_stat2.pivot(index=['metrics','n'],columns='method',values='p_value').reset_index()
df_stat = pd.concat([df_stat,df_stat2])

In [42]:
grouped1 = grouped.reset_index().copy()
grouped1['metrics'] = 'fitness'
grouped1 = grouped1.pivot(index=['metrics','n'],columns='method',values='fitness').reset_index()

grouped2 = grouped.reset_index().copy()
grouped2['metrics'] = 'run_time'
grouped2 = grouped2.pivot(index=['metrics','n'],columns='method',values='run_time').reset_index()

grouped3 = grouped.reset_index().copy()
grouped3['metrics'] = 'std_fitness'
grouped3 = grouped3.pivot(index=['metrics','n'],columns='method',values='std_fitness').reset_index()

grouped4 = grouped.reset_index().copy()
grouped4['metrics'] = 'std_run_time'
grouped4 = grouped4.pivot(index=['metrics','n'],columns='method',values='std_run_time').reset_index()

grouped = pd.concat([grouped1,grouped2,grouped3,grouped4])

In [43]:
save_to_excel_sheet(grouped,filename="dka_experiment_results.xlsx",sheet_name="various_n_pois_results")
save_to_excel_sheet(df_stat,filename="dka_experiment_results.xlsx",sheet_name="various_n_pois_p_value")